In [ ]:
import pandas as pd
import numpy as np
from scipy.interpolate import griddata
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from matplotlib.cm import ScalarMappable
import contextily as ctx

# mostra todas as colunas
pd.set_option("display.max_columns", None)


In [ ]:
import pandas as pd

csv_desc = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"

# Lê o CSV inteiro
csv_desc = pd.read_csv(csv_desc)
csv_desc

In [ ]:
csv_asc = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"

# Lê o CSV inteiro
csv_asc = pd.read_csv(csv_asc)
csv_asc

In [ ]:
csv_ortho = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"

# Lê o CSV inteiro
df_ortho = pd.read_csv(csv_ortho)
df_ortho

In [ ]:
# deslocamentos EGMS Alqueva
csv_desc = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
csv_asc = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"

In [ ]:
#df_nivel = pd.read_excel("data/nivel.xlsx")
df_nivel = pd.read_excel("data/alqueva_nivel.xlsx")
df_nivel

In [ ]:
#df_temp = pd.read_excel("data/nivel.xlsx")
df_temp = pd.read_excel("data/alqueva_temp.xlsx")
df_temp

apenas dados ortho

informação do egms

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from matplotlib.cm import ScalarMappable
import contextily as ctx
import numpy as np

# --- 1. Função para ler e filtrar CSV ---
def ler_e_filtrar(caminho_csv, norte_min=1855450, norte_max=1855650,
                  este_min=2792550, este_max=2792950):
    df = pd.read_csv(caminho_csv)
    required_cols = ['northing', 'easting']
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"CSV is missing column: {col}")
    df = df[
        (df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
        (df['easting'] >= este_min) & (df['easting'] <= este_max)
    ]
    return df

# --- 2. Carregar dados ORTHO ---
csv_ortho = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
df_ortho = ler_e_filtrar(csv_ortho)

# Agora vamos usar diretamente a coluna "mean_velocity"
componente = "mean_velocity"
vmin, vcenter, vmax = -20, 0, 20
usar_colormap_reverso = True

# --- 3. Criar GeoDataFrame ---
gdf = gpd.GeoDataFrame(
    df_ortho,
    geometry=gpd.points_from_xy(df_ortho['easting'], df_ortho['northing']),
    crs="EPSG:3035"
)
gdf_web = gdf.to_crs(epsg=3857)

# --- 4. Configurar colormap ---
cmap = plt.cm.jet
if usar_colormap_reverso:
    cmap = cmap.reversed()
norma = colors.TwoSlopeNorm(vmin=vmin, vcenter=vcenter, vmax=vmax)

# --- 5. Plotar ---
fig, ax = plt.subplots(figsize=(9, 6))
gdf_web.plot(
    ax=ax,
    column=componente,
    cmap=cmap,
    markersize=10,
    norm=norma,
    legend=False
)

# --- 6. Grelha 100x100 m ---
xmin, ymin, xmax, ymax = gdf_web.total_bounds
step = 100
for x in np.arange(xmin, xmax + step, step):
    ax.plot([x, x], [ymin, ymax], color='red', linewidth=0.8, alpha=0.8)
for y in np.arange(ymin, ymax + step, step):
    ax.plot([xmin, xmax], [y, y], color='red', linewidth=0.8, alpha=0.8)

# --- 7. Basemap ---
ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)
ax.set_axis_off()

# --- 8. Título e legenda ---
plt.title("Velocidade média (UP) - Dados ORTHO", fontsize=16)
sm = ScalarMappable(cmap=cmap, norm=norma)
sm._A = []
cbar = plt.colorbar(sm, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label("Velocidade (mm/ano)", fontsize=14)

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from matplotlib.cm import ScalarMappable
import contextily as ctx
import numpy as np

# --- 1. Função para ler e filtrar CSV ---
def ler_e_filtrar(caminho_csv, norte_min=1855450, norte_max=1855650,
                  este_min=2792550, este_max=2792950):
    df = pd.read_csv(caminho_csv)
    required_cols = ['northing', 'easting']
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"CSV is missing column: {col}")
    df = df[
        (df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
        (df['easting'] >= este_min) & (df['easting'] <= este_max)
    ]
    return df

# --- 2. Carregar dados ORTHO ---
csv_ortho = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
df_ortho = ler_e_filtrar(csv_ortho)

# Coluna com velocidade média
componente = "mean_velocity"
vmin, vcenter, vmax = -20, 0, 20
usar_colormap_reverso = True

# --- 3. Criar GeoDataFrame ---
gdf = gpd.GeoDataFrame(
    df_ortho,
    geometry=gpd.points_from_xy(df_ortho['easting'], df_ortho['northing']),
    crs="EPSG:3035"
)
gdf_web = gdf.to_crs(epsg=3857)

# --- 4. Configurar colormap ---
cmap = plt.cm.jet
if usar_colormap_reverso:
    cmap = cmap.reversed()
norma = colors.TwoSlopeNorm(vmin=vmin, vcenter=vcenter, vmax=vmax)

# --- 5. Plotar ---
fig, ax = plt.subplots(figsize=(12, 8))
gdf_web.plot(
    ax=ax,
    column=componente,
    cmap=cmap,
    markersize=50,
    norm=norma,
    legend=False
)

# --- Adicionar valores de velocidade em cada ponto ---
for x, y, vel in zip(gdf_web.geometry.x, gdf_web.geometry.y, gdf_web[componente]):
    ax.text(x, y, f"{vel:.1f}", fontsize=20, ha='center', va='bottom', color='black')

# --- 6. Grelha 100x100 m ---
xmin, ymin, xmax, ymax = gdf_web.total_bounds
step = 100
for x in np.arange(xmin, xmax + step, step):
    ax.plot([x, x], [ymin, ymax], color='red', linewidth=0.8, alpha=0.8)
for y in np.arange(ymin, ymax + step, step):
    ax.plot([xmin, xmax], [y, y], color='red', linewidth=0.8, alpha=0.8)

# --- 7. Basemap ---
ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)
ax.set_axis_off()

# --- 8. Título e legenda ---
plt.title("Velocidade média (UP) - Dados ORTHO", fontsize=16)
sm = ScalarMappable(cmap=cmap, norm=norma)
sm._A = []
cbar = plt.colorbar(sm, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label("Velocidade (mm/ano)", fontsize=14)

plt.tight_layout()
plt.show()


com melt. 

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from matplotlib.cm import ScalarMappable
import contextily as ctx
import seaborn as sns
import matplotlib.gridspec as gridspec
from mpl_toolkits.axes_grid1 import make_axes_locatable

# -------- 1. Função para ler e filtrar CSV --------
def ler_e_filtrar(caminho_csv, norte_min=1855450, norte_max=1855650,
                  este_min=2792550, este_max=2792950):
    df = pd.read_csv(caminho_csv)
    required_cols = ['northing', 'easting']
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"CSV is missing column: {col}")
    df = df[
        (df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
        (df['easting'] >= este_min) & (df['easting'] <= este_max)
    ]
    return df

# -------- 2. Carregar dados ORTHO --------
csv_ortho = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
df_ortho = ler_e_filtrar(csv_ortho)
componente = "up"
vmin, vcenter, vmax = -20, 0, 20
usar_colormap_reverso = True

# -------- 3. Melt das colunas de datas --------
def melt_ortho(df, comp):
    static_cols = df.columns[:13]
    date_cols = df.columns[13:]
    df_melt = df.melt(id_vars=static_cols, value_vars=date_cols,
                      var_name='date', value_name=comp)
    df_melt['date'] = pd.to_datetime(df_melt['date'], format='%Y%m%d')
    return df_melt

df_melt = melt_ortho(df_ortho, componente)

# -------- 4. GeoDataFrame geral --------
df_media = df_melt.groupby(['easting','northing'], as_index=False).mean(numeric_only=True)
gdf_all = gpd.GeoDataFrame(
    df_media,
    geometry=gpd.points_from_xy(df_media['easting'], df_media['northing']),
    crs="EPSG:3035"
)
gdf_web = gdf_all.to_crs(3857)
xmin, ymin, xmax, ymax = gdf_web.total_bounds

# -------- 5. Colormap --------
cmap = plt.cm.jet
if usar_colormap_reverso:
    cmap = cmap.reversed()
norm = colors.TwoSlopeNorm(vmin=vmin, vcenter=vcenter, vmax=vmax)

# -------- 6. Dados ambientais --------
df_nivel = pd.read_excel("data/alqueva_nivel.xlsx")
df_nivel['data'] = pd.to_datetime(df_nivel['data'])
df_nivel.set_index('data', inplace=True)

df_temp = pd.read_excel("data/alqueva_temp.xlsx")
df_temp['data'] = pd.to_datetime(df_temp['data'])
df_temp.set_index('data', inplace=True)

# -------- 7. Pontos a analisar (todos) --------
pontos = df_melt[['easting','northing']].drop_duplicates()

# -------- 8. Loop por ponto para gerar figuras --------
tabela_resultados = []

for _, row in pontos.iterrows():
    ponto = (row['easting'], row['northing'])
    serie = df_melt[(df_melt['easting']==ponto[0]) & (df_melt['northing']==ponto[1])].sort_values('date')
    
    datas = serie['date']
    up_values = serie[componente]
    
    nivel_alinhado = df_nivel.reindex(datas, method='nearest')['nivel'].values
    temp_alinhado = df_temp.reindex(datas, method='nearest')['med'].values
    
    valid = ~np.isnan(up_values) & ~np.isnan(nivel_alinhado) & ~np.isnan(temp_alinhado)
    if valid.sum() < 3:
        continue
    
    up_clean = up_values[valid]
    nivel_clean = nivel_alinhado[valid]
    temp_clean = temp_alinhado[valid]
    datas_clean = datas[valid]
    
    corr_nivel = np.corrcoef(up_clean, nivel_clean)[0,1]
    corr_temp = np.corrcoef(up_clean, temp_clean)[0,1]
    
    tabela_resultados.append({
        'easting': ponto[0],
        'northing': ponto[1],
        'correlacao_nivel': corr_nivel,
        'correlacao_temp': corr_temp
    })
    
    # --- Figura ---
    fig = plt.figure(figsize=(20,12))
    gs = gridspec.GridSpec(3,2, height_ratios=[2,1.2,1.2], width_ratios=[1.2,1])
    
    # --- Mapa ---
    ax_map = fig.add_subplot(gs[0,0])
    divider = make_axes_locatable(ax_map)
    cax = divider.append_axes("right", size="5%", pad=0.05)
    
    gdf_web.plot(ax=ax_map, column=componente, cmap=cmap, norm=norm, markersize=5, legend=True, cax=cax)
    gpd.GeoDataFrame(geometry=gpd.points_from_xy([ponto[0]],[ponto[1]]), crs="EPSG:3035").to_crs(3857).plot(
        ax=ax_map, color='red', edgecolor='black', markersize=100
    )
    
    # --- Grelha 100x100 m ---
    step = 100
    for x in np.arange(xmin, xmax+step, step):
        ax_map.plot([x,x],[ymin,ymax], color='red', linewidth=0.6, alpha=0.7)
    for y in np.arange(ymin, ymax+step, step):
        ax_map.plot([xmin,xmax],[y,y], color='red', linewidth=0.6, alpha=0.7)
    
    ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
    ax_map.set_xlim(xmin, xmax)
    ax_map.set_ylim(ymin, ymax)
    ax_map.set_title(f"Ponto ({ponto[0]:.0f}, {ponto[1]:.0f})")
    ax_map.set_axis_off()
    
    # --- Série temporal ---
    ax_series = fig.add_subplot(gs[0,1])
    ax_series.plot(datas_clean, up_clean, color="black", label="up (mm)")
    ax_series.axhline(0, color='gray', linestyle='--', linewidth=0.7)
    
    ax_nivel = ax_series.twinx()
    ax_nivel.plot(datas_clean, nivel_clean, color="blue", label="nível (m)")
    ax_nivel.set_ylabel("Nível (m)", color="blue")
    ax_nivel.tick_params(axis='y', labelcolor="blue")
    
    ax_temp = ax_series.twinx()
    ax_temp.spines["right"].set_position(("axes", 1.1))
    ax_temp.plot(datas_clean, temp_clean, color="red", label="temperatura (°C)")
    ax_temp.set_ylabel("Temp (°C)", color="red")
    ax_temp.tick_params(axis='y', labelcolor="red")
    
    ax_series.set_title("Série Temporal: up vs Nível vs Temperatura")
    ax_series.set_ylabel("Deslocamento (mm)")
    ax_series.set_xlabel("Data")
    ax_series.grid(True)
    
    # --- Dispersão up vs nível ---
    ax_corr_nivel = fig.add_subplot(gs[1,:])
    sns.scatterplot(x=up_clean, y=nivel_clean, color="black", label="up (mm)", ax=ax_corr_nivel)
    sns.regplot(x=up_clean, y=nivel_clean, scatter=False, color="blue", line_kws={"lw":2,"ls":"--"}, ax=ax_corr_nivel)
    ax_corr_nivel.set_title(f'Correlação up vs nível: r={corr_nivel:.2f}')
    ax_corr_nivel.set_xlabel("Deslocamento (mm)")
    ax_corr_nivel.set_ylabel("Nível (m)")
    ax_corr_nivel.grid(True)
    
    # --- Dispersão up vs temperatura ---
    ax_corr_temp = fig.add_subplot(gs[2,:])
    sns.scatterplot(x=up_clean, y=temp_clean, color="black", label="up (mm)", ax=ax_corr_temp)
    sns.regplot(x=up_clean, y=temp_clean, scatter=False, color="red", line_kws={"lw":2,"ls":"--"}, ax=ax_corr_temp)
    ax_corr_temp.set_title(f'Correlação up vs temperatura: r={corr_temp:.2f}')
    ax_corr_temp.set_xlabel("Deslocamento (mm)")
    ax_corr_temp.set_ylabel("Temperatura (°C)")
    ax_corr_temp.grid(True)
    
    plt.tight_layout()
    plt.show()

# -------- 9. Resultados --------
df_resultados = pd.DataFrame(tabela_resultados)
df_resultados.head()


In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx
import seaborn as sns
import matplotlib.gridspec as gridspec
import matplotlib.colors as colors


# -------- 1. Função para ler e filtrar CSV --------
def ler_e_filtrar(caminho_csv, norte_min=1855400, norte_max=1855700,
                  este_min=2792500, este_max=2793000):
    df = pd.read_csv(caminho_csv)
    required_cols = ['northing', 'easting']
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"CSV is missing column: {col}")
    df = df[
        (df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
        (df['easting'] >= este_min) & (df['easting'] <= este_max)
    ]
    return df

# -------- 2. Carregar dados ORTHO --------
csv_ortho = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
df_ortho = ler_e_filtrar(csv_ortho)
componente = "up"

# -------- 3. Melt das colunas de datas --------
def melt_ortho(df, comp):
    static_cols = df.columns[:13]
    date_cols = df.columns[13:]
    df_melt = df.melt(id_vars=static_cols, value_vars=date_cols,
                      var_name='date', value_name=comp)
    df_melt['date'] = pd.to_datetime(df_melt['date'], format='%Y%m%d')
    return df_melt

df_melt = melt_ortho(df_ortho, componente)

# -------- 4. GeoDataFrame geral --------
df_media = df_melt.groupby(['easting','northing'], as_index=False).mean(numeric_only=True)
gdf_all = gpd.GeoDataFrame(
    df_media,
    geometry=gpd.points_from_xy(df_media['easting'], df_media['northing']),
    crs="EPSG:3035"
)
gdf_web = gdf_all.to_crs(3857)
xmin, ymin, xmax, ymax = gdf_web.total_bounds

# -------- 5. Colormap InSAR --------
vmin, vmax = -20, 20
cmap_insar = plt.cm.jet
norm_insar = colors.Normalize(vmin=vmin, vmax=vmax)

# -------- 6. Dados ambientais --------
df_nivel = pd.read_excel("data/alqueva_nivel.xlsx")
df_nivel['data'] = pd.to_datetime(df_nivel['data'])
df_nivel.set_index('data', inplace=True)

df_temp = pd.read_excel("data/alqueva_temp.xlsx")
df_temp['data'] = pd.to_datetime(df_temp['data'])
df_temp.set_index('data', inplace=True)

# -------- 7. Pontos a analisar --------
pontos = df_melt[['easting','northing']].drop_duplicates()

# -------- 8. Loop --------
for _, row in pontos.iterrows():
    ponto = (row['easting'], row['northing'])
    serie = df_melt[(df_melt['easting']==ponto[0]) & (df_melt['northing']==ponto[1])].sort_values('date')
    
    datas = serie['date']
    up_values = serie[componente]
    
    nivel_alinhado = df_nivel.reindex(datas, method='nearest')['nivel'].values
    temp_alinhado = df_temp.reindex(datas, method='nearest')['med'].values
    
    valid = ~np.isnan(up_values) & ~np.isnan(nivel_alinhado) & ~np.isnan(temp_alinhado)
    if valid.sum() < 3:
        continue
    
    up_clean = up_values[valid]
    nivel_clean = nivel_alinhado[valid]
    temp_clean = temp_alinhado[valid]
    datas_clean = datas[valid]
    
    corr_nivel = np.corrcoef(up_clean, nivel_clean)[0,1]
    corr_temp = np.corrcoef(up_clean, temp_clean)[0,1]
    
    # --- Figura ---
    fig = plt.figure(figsize=(28,18))
    gs = gridspec.GridSpec(
        2, 3,
        height_ratios=[1.2, 2],
        width_ratios=[6, 1.2, 1.2],
        hspace=0.3,
        wspace=0.3
    )
    
    # --- Série temporal (em cima em toda a largura) ---
    ax_series = fig.add_subplot(gs[0,:])
    line_up, = ax_series.plot(datas_clean, up_clean, color="black", linewidth=2)
    ax_series.axhline(0, color='gray', linestyle='--', linewidth=0.7)
    
    ax_nivel = ax_series.twinx()
    line_nivel, = ax_nivel.plot(datas_clean, nivel_clean, color="blue", linewidth=2)
    ax_nivel.set_ylabel("Nível (m)", color="blue", fontsize=14)
    ax_nivel.tick_params(axis='y', labelcolor="blue", labelsize=12)
    
    ax_temp = ax_series.twinx()
    ax_temp.spines["right"].set_position(("axes", 1.05))  # mais próximo do eixo do nível
    line_temp, = ax_temp.plot(datas_clean, temp_clean, color="red", linewidth=2)
    ax_temp.set_ylabel("Temp (°C)", color="red", fontsize=14)
    ax_temp.tick_params(axis='y', labelcolor="red", labelsize=12)
    
    # Limites próximos para temperatura
    ax_temp.set_ylim(temp_clean.min() - 1, temp_clean.max() + 1)
    
    ax_series.set_title("Série Temporal", fontsize=20)
    ax_series.set_ylabel("Deslocamento (mm)", color="black", fontsize=14)
    ax_series.set_xlabel("Data", fontsize=14)
    ax_series.tick_params(axis='x', labelsize=12)
    ax_series.tick_params(axis='y', labelsize=12, labelcolor="black")
    ax_series.grid(True)
    
    # Legendas combinadas
    lines = [line_up, line_nivel, line_temp]
    labels = ["Deslocamento (mm)", "Nível (m)", "Temperatura (°C)"]
    ax_series.legend(lines, labels, loc="upper left", fontsize=12, frameon=False)
    
    # --- Mapa ---
    ax_map = fig.add_subplot(gs[1,0])
    sc = gdf_web.plot(ax=ax_map, column=componente, cmap=cmap_insar, norm=norm_insar, markersize=80, legend=False)
    gpd.GeoDataFrame(
        geometry=gpd.points_from_xy([ponto[0]],[ponto[1]]),
        crs="EPSG:3035"
    ).to_crs(3857).plot(ax=ax_map, color='red', edgecolor='black', markersize=300)
    
    ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
    ax_map.set_xlim(xmin, xmax)
    ax_map.set_ylim(ymin, ymax)
    ax_map.set_title(f"Ponto ({ponto[0]:.0f}, {ponto[1]:.0f})", fontsize=18)
    ax_map.set_axis_off()
    
    # Legenda de cores InSAR dentro do mapa
    sm = plt.cm.ScalarMappable(cmap=cmap_insar, norm=norm_insar)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax_map, fraction=0.046, pad=0.02)
    cbar.set_label("Deslocamento InSAR (mm)", fontsize=14)
    
    # --- Correlação up vs nível ---
    ax_corr_nivel = fig.add_subplot(gs[1,1])
    sns.scatterplot(x=up_clean, y=nivel_clean, color="black", s=80, ax=ax_corr_nivel)
    sns.regplot(x=up_clean, y=nivel_clean, scatter=False, color="blue",
                line_kws={"lw":2,"ls":"--"}, ax=ax_corr_nivel)
    ax_corr_nivel.set_title(f'up vs nível: r={corr_nivel:.2f}', fontsize=16)
    ax_corr_nivel.set_xlabel("Deslocamento (mm)", fontsize=14)
    ax_corr_nivel.set_ylabel("Nível (m)", fontsize=14)
    ax_corr_nivel.tick_params(axis='both', labelsize=12)
    ax_corr_nivel.grid(True)
    
    # --- Correlação up vs temperatura ---
    ax_corr_temp = fig.add_subplot(gs[1,2])
    sns.scatterplot(x=up_clean, y=temp_clean, color="black", s=80, ax=ax_corr_temp)
    sns.regplot(x=up_clean, y=temp_clean, scatter=False, color="red",
                line_kws={"lw":2,"ls":"--"}, ax=ax_corr_temp)
    ax_corr_temp.set_title(f'up vs temperatura: r={corr_temp:.2f}', fontsize=16)
    ax_corr_temp.set_xlabel("Deslocamento (mm)", fontsize=14)
    ax_corr_temp.set_ylabel("Temperatura (°C)", fontsize=14)
    ax_corr_temp.tick_params(axis='both', labelsize=12)
    ax_corr_temp.grid(True)
    
    plt.tight_layout()
    plt.show()


- clusters foram definidos espacialmente:
- Ou seja, o KMeans agrupou os pontos apenas pela localização no espaço (x, y no EPSG:3857).
Isso significa que dois pontos próximos ficam no mesmo cluster, mesmo que as séries de deslocamento sejam muito diferentes.

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx
from sklearn.cluster import KMeans
from matplotlib import cm
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec


# -------- 1. Função para ler e filtrar CSV --------
def ler_e_filtrar(caminho_csv, norte_min=1855050, norte_max=1855850,
                  este_min=2792250, este_max=2793250):
    df = pd.read_csv(caminho_csv)
    required_cols = ['northing', 'easting']
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"CSV is missing column: {col}")
    df = df[
        (df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
        (df['easting'] >= este_min) & (df['easting'] <= este_max)
    ]
    return df

# -------- 2. Carregar dados ORTHO --------
csv_ortho = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
df_ortho = ler_e_filtrar(csv_ortho)
componente = "up"

# -------- 3. Melt das colunas de datas --------
def melt_ortho(df, comp):
    static_cols = df.columns[:13]
    date_cols = df.columns[13:]
    df_melt = df.melt(id_vars=static_cols, value_vars=date_cols,
                      var_name='date', value_name=comp)
    df_melt['date'] = pd.to_datetime(df_melt['date'], format='%Y%m%d')
    return df_melt

df_melt = melt_ortho(df_ortho, componente)

# -------- 4. Pivot para matriz temporal --------
# Linhas = pontos, colunas = datas
serie_matrix = df_melt.pivot_table(
    index=['easting','northing'],
    columns='date',
    values='up'
)

tempo = serie_matrix.columns

# -------- 5. Criar GeoDataFrame para clustering e mapa --------
gdf = gpd.GeoDataFrame(
    serie_matrix.reset_index()[['easting','northing']],
    geometry=gpd.points_from_xy(
        serie_matrix.reset_index().easting,
        serie_matrix.reset_index().northing
    ),
    crs="EPSG:3035"
)
gdf_web = gdf.to_crs(epsg=3857)

# -------- 6. KMeans clustering espacial --------
coords = np.vstack([gdf_web.geometry.x, gdf_web.geometry.y]).T
n_clusters = 4
kmeans = KMeans(n_clusters=n_clusters, random_state=0, n_init=10).fit(coords)
gdf_web["cluster"] = kmeans.labels_
serie_matrix["cluster"] = kmeans.labels_

# -------- 7. Definir escala Y global para todos os plots --------
global_min = np.nanmin(serie_matrix.drop(columns="cluster").values)
global_max = np.nanmax(serie_matrix.drop(columns="cluster").values)

# -------- 8. Layout dos plots --------
ncols = 2
nrows = int(np.ceil(n_clusters / ncols))
fig = plt.figure(figsize=(22, 6 + 3*nrows))
gs = gridspec.GridSpec(nrows+1, ncols, height_ratios=[2]+[1]*nrows, hspace=0.4, wspace=0.25)

# -------- 9. Mapa dos clusters (linha de cima) --------
ax_mapa = fig.add_subplot(gs[0, :])
gdf_web.plot(ax=ax_mapa, column="cluster", categorical=True,
             legend=True, cmap="tab10", markersize=60,
             alpha=0.9, edgecolor="black")
ctx.add_basemap(ax_mapa, source=ctx.providers.Esri.WorldImagery)
ax_mapa.set_axis_off()
ax_mapa.set_title(f"Clusters espaciais (KMeans, {n_clusters} clusters)", fontsize=18)

# -------- 10. Paleta de cores para clusters --------
cmap = cm.get_cmap('tab10', n_clusters)
cores_clusters = [cmap(i) for i in range(n_clusters)]

# -------- 11. Plots das séries temporais por cluster --------
for i in range(n_clusters):
    row = (i // ncols) + 1
    col = i % ncols
    ax = fig.add_subplot(gs[row, col])
    
    idx = serie_matrix["cluster"] == i
    series_cluster = serie_matrix[idx].drop(columns="cluster").values
    
    # Séries individuais em cinza
    for serie in series_cluster:
        ax.plot(tempo, serie, color="lightgray", linewidth=0.8, alpha=0.7)
    
    # Linha média do cluster
    media_cluster = np.nanmean(series_cluster, axis=0)
    ax.plot(tempo, media_cluster, color=cores_clusters[i], linewidth=3,
            label=f"Média Cluster {i}")
    
    ax.set_title(f"Cluster {i} ({series_cluster.shape[0]} séries)", fontsize=14)
    ax.grid(True, linestyle="--", alpha=0.6)
    ax.legend(fontsize=10)
    ax.set_ylabel("Deslocamento (mm)")
    ax.set_ylim(global_min, global_max)  # mesma escala para todos
    
    # Eixo de datas legível
    ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=4, maxticks=8))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))

# -------- 12. Apenas a última linha tem xlabel --------
for ax in fig.get_axes()[-ncols:]:
    ax.set_xlabel("Tempo", fontsize=12)

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


- clustering pelo tipo de deslocamento (padrão temporal):
- O que temos de fazer é aplicar o KMeans às séries temporais normalizadas, em vez das coordenadas. Assim, os pontos são agrupados por semelhança de comportamento no tempo (independentemente da localização).

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from matplotlib import cm
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec


# -------- 1. Função para ler e filtrar CSV --------
def ler_e_filtrar(caminho_csv, norte_min=1855050, norte_max=1855850,
                  este_min=2792250, este_max=2793250):
    df = pd.read_csv(caminho_csv)
    required_cols = ['northing', 'easting']
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"CSV is missing column: {col}")
    df = df[
        (df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
        (df['easting'] >= este_min) & (df['easting'] <= este_max)
    ]
    return df

# -------- 2. Carregar dados --------
csv_ortho = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
df = ler_e_filtrar(csv_ortho)

# Derreter colunas temporais -> formato longo
df_melt = df.melt(id_vars=["easting", "northing"], var_name="date", value_name="up")
df_melt["date"] = pd.to_datetime(df_melt["date"], errors="coerce")

# Pivot: linhas = pontos, colunas = datas
serie_matrix = df_melt.pivot_table(
    index=['easting','northing'],
    columns='date',
    values='up'
)

tempo = serie_matrix.columns

# -------- 3. Clustering temporal --------
X = serie_matrix.values
X_scaled = StandardScaler().fit_transform(X)

n_clusters = 4
kmeans = KMeans(n_clusters=n_clusters, random_state=0, n_init=10).fit(X_scaled)
serie_matrix["cluster"] = kmeans.labels_

# Criar GeoDataFrame
gdf = gpd.GeoDataFrame(
    serie_matrix.reset_index()[['easting','northing']],
    geometry=gpd.points_from_xy(
        serie_matrix.reset_index().easting,
        serie_matrix.reset_index().northing
    ),
    crs="EPSG:3035"
)
gdf["cluster"] = kmeans.labels_
gdf_web = gdf.to_crs(epsg=3857)

# -------- 4. Layout --------
ncols = 2
nrows = int(np.ceil(n_clusters / ncols))

fig = plt.figure(figsize=(22, 6 + 3 * nrows))
gs = gridspec.GridSpec(nrows + 1, ncols,
                       height_ratios=[2] + [1]*nrows,
                       hspace=0.4, wspace=0.25)

# -------- 5. Mapa --------
ax_mapa = fig.add_subplot(gs[0, :])
gdf_web.plot(ax=ax_mapa, column="cluster", categorical=True,
             legend=True, cmap="tab10", markersize=60,
             alpha=0.9, edgecolor="black")
ctx.add_basemap(ax_mapa, source=ctx.providers.Esri.WorldImagery)
ax_mapa.set_axis_off()
ax_mapa.set_title(f"Clusters temporais (KMeans, {n_clusters} clusters)", fontsize=18)

# -------- 6. Escala comum --------
global_min = np.nanmin(serie_matrix.drop(columns="cluster").values)
global_max = np.nanmax(serie_matrix.drop(columns="cluster").values)

# -------- 7. Séries por cluster --------
cmap = cm.get_cmap('tab10', n_clusters)
cores_clusters = [cmap(i) for i in range(n_clusters)]

for i in range(n_clusters):
    row = (i // ncols) + 1
    col = i % ncols
    ax = fig.add_subplot(gs[row, col])
    
    idx = serie_matrix["cluster"] == i
    series_cluster = serie_matrix[idx].drop(columns="cluster").values
    
    # Séries individuais
    for serie in series_cluster:
        ax.plot(tempo, serie, color="lightgray", linewidth=0.8, alpha=0.7)
    
    # Média
    media_cluster = np.nanmean(series_cluster, axis=0)
    ax.plot(tempo, media_cluster, color=cores_clusters[i], linewidth=3,
            label=f"Média Cluster {i}")
    
    ax.set_title(f"Cluster {i} ({series_cluster.shape[0]} séries)", fontsize=14)
    ax.grid(True, linestyle="--", alpha=0.6)
    ax.legend(fontsize=10)
    ax.set_ylabel("Deslocamento (mm)")
    
    # Mesma escala
    ax.set_ylim(global_min, global_max)
    
    # Datas
    ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=4, maxticks=8))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))

# Eixo X só na última linha
for ax in fig.get_axes()[-ncols:]:
    ax.set_xlabel("Tempo", fontsize=12)

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()



calibrated

informação do egms

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from matplotlib.cm import ScalarMappable
import contextily as ctx
import numpy as np


# --- 1. Função para ler e filtrar CSV ---
def ler_e_filtrar(caminho_csv, norte_min=1855050, norte_max=1855850,
                  este_min=2792250, este_max=2793250):
    df = pd.read_csv(caminho_csv)
    required_cols = ['northing', 'easting']
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"CSV is missing column: {col}")
    df = df[
        (df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
        (df['easting'] >= este_min) & (df['easting'] <= este_max)
    ]
    return df

# --- 2. Carregar dados calibrated DESC ---
csv_desc = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
df_desc = ler_e_filtrar(csv_desc)

# Aqui usamos diretamente a coluna mean_velocity (ou outra coluna de interesse)
componente = "mean_velocity"  
vmin, vcenter, vmax = -20, 0, 20
usar_colormap_reverso = True

# --- 3. Criar GeoDataFrame ---
gdf = gpd.GeoDataFrame(
    df_desc,
    geometry=gpd.points_from_xy(df_desc['easting'], df_desc['northing']),
    crs="EPSG:3035"
)
gdf_web = gdf.to_crs(epsg=3857)

# --- 4. Configurar colormap ---
cmap = plt.cm.jet
if usar_colormap_reverso:
    cmap = cmap.reversed()
norma = colors.TwoSlopeNorm(vmin=vmin, vcenter=vcenter, vmax=vmax)

# --- 5. Plotar ---
fig, ax = plt.subplots(figsize=(9, 6))
gdf_web.plot(
    ax=ax,
    column=componente,
    cmap=cmap,
    markersize=10,
    norm=norma,
    legend=False
)

# --- 6. Grelha 100x100 m ---
xmin, ymin, xmax, ymax = gdf_web.total_bounds
step = 100
for x in np.arange(xmin, xmax + step, step):
    ax.plot([x, x], [ymin, ymax], color='red', linewidth=0.8, alpha=0.8)
for y in np.arange(ymin, ymax + step, step):
    ax.plot([xmin, xmax], [y, y], color='red', linewidth=0.8, alpha=0.8)

# --- 7. Basemap ---
ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)
ax.set_axis_off()

# --- 8. Título e legenda ---
plt.title(f"Velocidade média ({componente}) - Dados Calibrated DESC", fontsize=16)
sm = ScalarMappable(cmap=cmap, norm=norma)
sm._A = []
cbar = plt.colorbar(sm, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label("Velocidade LOS (mm/ano)", fontsize=14)

plt.tight_layout()
plt.show()


com melp. calcula o up

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from matplotlib.cm import ScalarMappable
import contextily as ctx
import numpy as np

# --- 1. Função para ler e filtrar CSV ---
def ler_e_filtrar(caminho_csv, norte_min=1855050, norte_max=1855850,
                  este_min=2792250, este_max=2793250):
    df = pd.read_csv(caminho_csv)
    required_cols = ['northing', 'easting']
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"CSV is missing column: {col}")
    df = df[
        (df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
        (df['easting'] >= este_min) & (df['easting'] <= este_max)
    ]
    return df


# --- 2. Carregar dados calibrated ASC ---
csv_desc = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
df_desc = ler_e_filtrar(csv_desc)
componente = "los"   # aqui é linha de visada
vmin, vcenter, vmax = -20, 0, 20
usar_colormap_reverso = True

# --- 3. Melt das colunas de datas ---
def melt_calibrated(df, comp):
    static_cols = df.columns[:24]  # primeiras 24 colunas = metadados
    date_cols = df.columns[24:]    # resto = datas
    df_melt = df.melt(id_vars=static_cols, value_vars=date_cols,
                      var_name='date', value_name=comp)
    df_melt['date'] = pd.to_datetime(df_melt['date'], format='%Y%m%d')
    return df_melt

df_melt = melt_calibrated(df_desc, componente)

# --- 4. Média por ponto ---
df_media = df_melt.groupby(['easting', 'northing'], as_index=False).mean(numeric_only=True)

# --- 5. Criar GeoDataFrame ---
gdf = gpd.GeoDataFrame(
    df_media,
    geometry=gpd.points_from_xy(df_media['easting'], df_media['northing']),
    crs="EPSG:3035"
)
gdf_web = gdf.to_crs(epsg=3857)

# --- 6. Configurar colormap ---
cmap = plt.cm.jet
if usar_colormap_reverso:
    cmap = cmap.reversed()
norma = colors.TwoSlopeNorm(vmin=vmin, vcenter=vcenter, vmax=vmax)

# --- 7. Plotar ---
fig, ax = plt.subplots(figsize=(9, 6))
gdf_web.plot(
    ax=ax,
    column=componente,
    cmap=cmap,
    markersize=10,
    norm=norma,
    legend=False
)

# --- 8. Grelha 100x100 m ---
xmin, ymin, xmax, ymax = gdf_web.total_bounds
step = 100
for x in np.arange(xmin, xmax + step, step):
    ax.plot([x, x], [ymin, ymax], color='red', linewidth=0.8, alpha=0.8)
for y in np.arange(ymin, ymax + step, step):
    ax.plot([xmin, xmax], [y, y], color='red', linewidth=0.8, alpha=0.8)

# --- 9. Basemap ---
ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)
ax.set_axis_off()

# --- 10. Título e legenda ---
plt.title(f"Deslocamento médio ({componente}) - Dados Calibrated ASC", fontsize=16)
sm = ScalarMappable(cmap=cmap, norm=norma)
sm._A = []
cbar = plt.colorbar(sm, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label("Deslocamento LOS (mm)", fontsize=14)

plt.tight_layout()
plt.show()


- clusters foram definidos espacialmente:
- Ou seja, o KMeans agrupou os pontos apenas pela localização no espaço (x, y no EPSG:3857).
Isso significa que dois pontos próximos ficam no mesmo cluster, mesmo que as séries de deslocamento sejam muito diferentes.

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx
from sklearn.cluster import KMeans
from matplotlib import cm
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec

# -------- 1. Função para ler e filtrar CSV --------
def ler_e_filtrar(caminho_csv, norte_min=1855050, norte_max=1855850,
                  este_min=2792250, este_max=2793250):
    df = pd.read_csv(caminho_csv)
    required_cols = ['northing', 'easting']
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"CSV is missing column: {col}")
    df = df[
        (df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
        (df['easting'] >= este_min) & (df['easting'] <= este_max)
    ]
    return df


# -------- 2. Carregar dados ASC/DESC --------
csv_desc = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
df_asc = ler_e_filtrar(csv_desc)
componente = "up"

# -------- 3. Melt das colunas de datas --------
def melt_asc(df, comp):
    static_cols = df.columns[:24]  # ajusta se necessário
    date_cols = df.columns[24:]
    df_melt = df.melt(id_vars=static_cols, value_vars=date_cols,
                      var_name='date', value_name=comp)
    df_melt['date'] = pd.to_datetime(df_melt['date'], format='%Y%m%d')
    return df_melt

df_melt = melt_asc(df_asc, componente)

# -------- 4. Pivot para matriz temporal --------
serie_matrix = df_melt.pivot_table(
    index=['easting','northing'],
    columns='date',
    values='up'
)
tempo = serie_matrix.columns

# -------- 5. Criar GeoDataFrame --------
gdf = gpd.GeoDataFrame(
    serie_matrix.reset_index()[['easting','northing']],
    geometry=gpd.points_from_xy(
        serie_matrix.reset_index().easting,
        serie_matrix.reset_index().northing
    ),
    crs="EPSG:3035"
)
gdf_web = gdf.to_crs(epsg=3857)

# -------- 6. KMeans clustering espacial --------
coords = np.vstack([gdf_web.geometry.x, gdf_web.geometry.y]).T
n_clusters = 4
kmeans = KMeans(n_clusters=n_clusters, random_state=0, n_init=10).fit(coords)
gdf_web["cluster"] = kmeans.labels_
serie_matrix["cluster"] = kmeans.labels_

# -------- 7. Definir escala Y global --------
global_min = np.nanmin(serie_matrix.drop(columns="cluster").values)
global_max = np.nanmax(serie_matrix.drop(columns="cluster").values)

# -------- 8. Layout dos plots --------
ncols = 2
nrows = int(np.ceil(n_clusters / ncols))
fig = plt.figure(figsize=(22, 6 + 3*nrows))
gs = gridspec.GridSpec(nrows+1, ncols, height_ratios=[3]+[1]*nrows, hspace=0.4, wspace=0.25)

# -------- 9. Mapa dos clusters (linha de cima) --------
ax_mapa = fig.add_subplot(gs[0, :])
gdf_web.plot(ax=ax_mapa, column="cluster", categorical=True,
             legend=True, cmap="tab10", markersize=60,
             alpha=0.9, edgecolor="black")
ctx.add_basemap(ax_mapa, source=ctx.providers.Esri.WorldImagery)
ax_mapa.set_axis_off()
ax_mapa.set_title(f"Clusters espaciais (KMeans, {n_clusters} clusters)", fontsize=18)

# -------- 10. Paleta de cores para clusters --------
cmap = cm.get_cmap('tab10', n_clusters)
cores_clusters = [cmap(i) for i in range(n_clusters)]

# -------- 11. Plots das séries temporais por cluster --------
for i in range(n_clusters):
    row = (i // ncols) + 1
    col = i % ncols
    ax = fig.add_subplot(gs[row, col])
    
    idx = serie_matrix["cluster"] == i
    series_cluster = serie_matrix[idx].drop(columns="cluster").values
    
    # Séries individuais em cinza
    for serie in series_cluster:
        ax.plot(tempo, serie, color="lightgray", linewidth=0.8, alpha=0.7)
    
    # Linha média do cluster
    media_cluster = np.nanmean(series_cluster, axis=0)
    ax.plot(tempo, media_cluster, color=cores_clusters[i], linewidth=3,
            label=f"Média Cluster {i}")
    
    ax.set_title(f"Cluster {i} ({series_cluster.shape[0]} séries)", fontsize=14)
    ax.grid(True, linestyle="--", alpha=0.6)
    ax.legend(fontsize=10)
    ax.set_ylabel("Deslocamento (mm)")
    ax.set_ylim(global_min, global_max)  # mesma escala para todos
    
    # Eixo de datas legível
    ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=4, maxticks=8))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))

# -------- 12. Apenas a última linha tem xlabel --------
for ax in fig.get_axes()[-ncols:]:
    ax.set_xlabel("Tempo", fontsize=12)

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


sub-clustering

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from matplotlib import cm
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec

# -------- 1. Função para ler e filtrar CSV --------
def ler_e_filtrar(caminho_csv, norte_min=1855050, norte_max=1855850,
                  este_min=2792250, este_max=2793250):
    df = pd.read_csv(caminho_csv)
    required_cols = ['northing', 'easting']
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"CSV is missing column: {col}")
    df = df[
        (df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
        (df['easting'] >= este_min) & (df['easting'] <= este_max)
    ]
    return df


# -------- 2. Carregar dados ASC/DESC --------
csv_asc = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
df_asc = ler_e_filtrar(csv_asc)
componente = "up"

# -------- 3. Melt das colunas de datas --------
def melt_asc(df, comp):
    static_cols = df.columns[:24]  # ajustar conforme necessário
    date_cols = df.columns[24:]
    df_melt = df.melt(id_vars=static_cols, value_vars=date_cols,
                      var_name='date', value_name=comp)
    df_melt['date'] = pd.to_datetime(df_melt['date'], format='%Y%m%d')
    return df_melt

df_melt = melt_asc(df_asc, componente)

# -------- 4. Pivot para matriz temporal --------
serie_matrix = df_melt.pivot_table(
    index=['easting','northing'],
    columns='date',
    values='up'
)
tempo = serie_matrix.columns

# -------- 5. Criar GeoDataFrame --------
gdf = gpd.GeoDataFrame(
    serie_matrix.reset_index()[['easting','northing']],
    geometry=gpd.points_from_xy(
        serie_matrix.reset_index().easting,
        serie_matrix.reset_index().northing
    ),
    crs="EPSG:3035"
)
gdf_web = gdf.to_crs(epsg=3857)

# -------- 6. KMeans clustering espacial --------
coords = np.vstack([gdf_web.geometry.x, gdf_web.geometry.y]).T
n_clusters = 4
kmeans = KMeans(n_clusters=n_clusters, random_state=0, n_init=10).fit(coords)
gdf_web["cluster"] = kmeans.labels_
serie_matrix["cluster"] = kmeans.labels_

# -------- 7. Escolher um cluster espacial --------
cluster_escolhido = 3  # exemplo
serie_matrix_reset = serie_matrix.reset_index()
subset_idx = serie_matrix_reset["cluster"] == cluster_escolhido

# Subset das séries temporais e GeoDataFrame
subset_matrix = serie_matrix_reset[subset_idx].drop(columns="cluster")
gdf_web_reset = gdf_web.reset_index(drop=True)
subset_gdf = gdf_web_reset[subset_idx.values]

# -------- 8. Preencher NaNs e alinhar --------
subset_matrix_filled = subset_matrix.copy()
subset_matrix_filled = subset_matrix_filled.reindex(columns=tempo)  # garante mesmas colunas
subset_matrix_filled = subset_matrix_filled.fillna(method='ffill', axis=1).fillna(method='bfill', axis=1)

# -------- 9. KMeans temporal --------
X = subset_matrix_filled.values
X_scaled = StandardScaler().fit_transform(X)
n_clusters_temporal = 4
kmeans_temporal = KMeans(n_clusters=n_clusters_temporal, random_state=0, n_init=10).fit(X_scaled)
subset_matrix_filled["cluster_temporal"] = kmeans_temporal.labels_
subset_gdf["cluster_temporal"] = kmeans_temporal.labels_

# -------- 10. Escala Y global --------
global_min = np.nanmin(subset_matrix_filled.drop(columns="cluster_temporal").values)
global_max = np.nanmax(subset_matrix_filled.drop(columns="cluster_temporal").values)

# -------- 11. Layout dos plots --------
ncols = 2
nrows = int(np.ceil(n_clusters_temporal / ncols))
fig = plt.figure(figsize=(20, 6 + 3*nrows))
gs = gridspec.GridSpec(nrows+1, ncols, height_ratios=[4]+[1]*nrows, hspace=0.4, wspace=0.25)

# -------- 12. Mapa completo com destaque do cluster escolhido --------
ax_mapa = fig.add_subplot(gs[0, :])
# Todos os pontos em cinza
gdf_web.plot(ax=ax_mapa, color="lightgray", markersize=20, alpha=0.5)
# Pontos do cluster escolhido coloridos por sub-cluster
subset_gdf.plot(ax=ax_mapa, column="cluster_temporal", categorical=True,
                legend=True, cmap="tab10", markersize=60, alpha=0.9, edgecolor="black")
ctx.add_basemap(ax_mapa, source=ctx.providers.Esri.WorldImagery)
ax_mapa.set_axis_off()
ax_mapa.set_title(f"Sub-clusters temporais (Cluster {cluster_escolhido})", fontsize=18)

# -------- 13. Paleta de cores --------
cmap = cm.get_cmap('tab10', n_clusters_temporal)
cores_clusters = [cmap(i) for i in range(n_clusters_temporal)]

# -------- 14. Plots das séries temporais (cinza + média, sem grade) --------
for i in range(n_clusters_temporal):
    row = (i // ncols) + 1
    col = i % ncols
    ax = fig.add_subplot(gs[row, col])
    
    idx = subset_matrix_filled["cluster_temporal"] == i
    series_cluster = subset_matrix_filled[idx].drop(columns="cluster_temporal").values
    
    # Séries individuais em cinza
    for serie in series_cluster:
        ax.plot(tempo, serie, color="lightgray", linewidth=0.8, alpha=0.7)
    
    # Linha média do sub-cluster
    media_cluster = np.nanmean(series_cluster, axis=0)
    ax.plot(tempo, media_cluster, color=cores_clusters[i], linewidth=3,
            label=f"Média Sub-cluster {i}")
    
    ax.set_title(f"Sub-cluster {i} ({series_cluster.shape[0]} séries)", fontsize=14)
    
    # Remover linhas de grade
    ax.grid(False)
    
    #ax.legend(fontsize=10)
    ax.set_ylabel("Deslocamento (mm)")
    ax.set_ylim(global_min, global_max)
    
    ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=4, maxticks=8))
    #ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

# -------- 15. Apenas a última linha tem xlabel --------
for ax in fig.get_axes()[-ncols:]:
    ax.set_xlabel("Tempo", fontsize=12)

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


1. Decomposição sazonal (seasonal decompose)

O que é:
Separar cada série temporal em três componentes:
- Tendência (trend): movimento de longo prazo (deslocamento contínuo da barragem ou encosta).
- Sazonalidade (seasonal): variações periódicas (ex.: devido a enchentes, ciclos de chuva/estação seca).
- Resíduos (resid): variações aleatórias ou ruído.

Como usar:
- Aplicar statsmodels.tsa.seasonal_decompose ou STL decomposition à série média de cada cluster ou sub-cluster.
- Identificar padrões regulares que podem ser causados por:
    - Mudança no nível de água da barragem.
    - Expansão/contração térmica.
    - Ciclos sazonais de vegetação ou solo.

Benefício:
- Facilita a distinção entre movimento real de longo prazo e variações sazonais esperadas, ajudando a detectar instabilidade anômala.

# 1. O que o STL faz

O STL é uma técnica de decomposição de séries temporais que separa a série original em três componentes principais:

## Trend (Tendência)

- Representa o movimento de longo prazo da série.  
- Ex.: se tens deslocamento do solo ao longo de anos, a tendência mostra se está subindo ou descendo de forma gradual.

## Seasonal (Sazonalidade)

- Mostra padrões repetitivos e cíclicos ao longo do tempo.  
- Ex.: variações anuais ou mensais, como inchaço do solo na estação das chuvas ou evaporação sazonal.

## Residual (Resíduos / Ruído)

- O que sobra depois de remover tendência e sazonalidade.  
- Contém flutuações aleatórias, erros de medição ou eventos inesperados.

Matematicamente, a série original \(Y_t\) é decomposta assim:

\[
Y_t = Trend_t + Seasonal_t + Residual_t
\]


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import STL

# --- Assumimos subset_matrix_filled e tempo já definidos ---
n_clusters_temporal = subset_matrix_filled["cluster_temporal"].nunique()

# Calcular séries médias de todos os sub-clusters
medias_clusters = []
for i in range(n_clusters_temporal):
    idx = subset_matrix_filled["cluster_temporal"] == i
    series_cluster = subset_matrix_filled[idx].drop(columns="cluster_temporal").values
    medias_clusters.append(np.nanmean(series_cluster, axis=0))

# Obter limites globais Y
global_min = np.nanmin(medias_clusters)
global_max = np.nanmax(medias_clusters)

# Layout: 2 colunas
ncols = 2
nrows = int(np.ceil(n_clusters_temporal / ncols))

fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(16, 4*nrows))
axes = axes.flatten()

for i in range(n_clusters_temporal):
    # Série média
    ts = pd.Series(medias_clusters[i], index=tempo)
    ts_interp = ts.interpolate()
    
    # Decomposição STL
    stl = STL(ts_interp, period=61)  # ~6 em 6 dias → ~61 pontos/ano
    res = stl.fit()
    
    # Plot no subplot
    ax = axes[i]
    ax.plot(ts_interp.index, ts_interp, color="black", label="Original", alpha=0.6)
    ax.plot(ts_interp.index, res.trend, color="blue", linewidth=2, label="Tendência")
    ax.plot(ts_interp.index, res.seasonal, color="green", linewidth=2, label="Sazonalidade")
    ax.plot(ts_interp.index, res.resid, color="red", linewidth=1, linestyle="--", label="Resíduo")
    
    ax.set_title(f"Sub-cluster {i} - Decomposição STL", fontsize=12, weight="bold")
    ax.legend(fontsize=8)
    
    # Aplicar escala Y global
    ax.set_ylim(global_min, global_max)
    
    # Remover linhas auxiliares
    ax.grid(False)

# Se sobrar espaço vazio (número ímpar de clusters), remover eixos
for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()




In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import STL

# --- Assumimos subset_matrix_filled e tempo já definidos ---
n_clusters_temporal = subset_matrix_filled["cluster_temporal"].nunique()

# Calcular séries médias de todos os sub-clusters
medias_clusters = []
for i in range(n_clusters_temporal):
    idx = subset_matrix_filled["cluster_temporal"] == i
    series_cluster = subset_matrix_filled[idx].drop(columns="cluster_temporal").values
    medias_clusters.append(np.nanmean(series_cluster, axis=0))

# Fazer decomposição STL de cada média
decomps = []
for serie in medias_clusters:
    ts = pd.Series(serie, index=tempo).interpolate()
    stl = STL(ts, period=61)  # ~6 em 6 dias → ~61 pontos/ano
    res = stl.fit()
    decomps.append(res)

# Escala global única (mínimo e máximo de todas as componentes)
global_min = min([
    np.nanmin(np.concatenate([
        d.observed, d.trend, d.seasonal, d.resid
    ])) for d in decomps
])
global_max = max([
    np.nanmax(np.concatenate([
        d.observed, d.trend, d.seasonal, d.resid
    ])) for d in decomps
])

# Layout: 4 linhas (componentes) × clusters (colunas)
fig, axes = plt.subplots(
    nrows=4, ncols=n_clusters_temporal, figsize=(4*n_clusters_temporal, 8),
    sharex=True
)

# Garantir que axes é 2D
if n_clusters_temporal == 1:
    axes = axes[:, np.newaxis]

componentes = ["original", "trend", "seasonal", "resid"]

for j, res in enumerate(decomps):  # para cada cluster
    ts_index = tempo
    series = {
        "original": res.observed,
        "trend": res.trend,
        "seasonal": res.seasonal,
        "resid": res.resid
    }
    for i, comp in enumerate(componentes):
        ax = axes[i, j]
        ax.plot(ts_index, series[comp], color="black", linewidth=1.8)
        ax.set_ylim(global_min, global_max)  # mesma escala para tudo
        ax.grid(False)
        if j == 0:
            ax.set_ylabel(comp.capitalize(), fontsize=12)
        if i == 0:
            ax.set_title(f"Sub-cluster {j}", fontsize=12)

plt.tight_layout()
plt.show()


2. Correlação com outras variáveis

O que é:
Comparar a série média do cluster com variáveis externas, como:

- Nível de água da barragem.
- Precipitação.
- Temperatura média.
- Operações da barragem (enchimentos, liberações).

Como usar:
- Calcular correlação linear (np.corrcoef) ou cross-correlation para identificar atrasos temporais.
- Pode ser feito por cluster ou sub-cluster para ver quais zonas respondem mais a cada variável.

Benefício:
- Permite atribuir causas ao movimento, diferenciar instabilidades naturais de mudanças induzidas por operação da barragem.

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec


# -------- 1. Função para ler e filtrar CSV --------
def ler_e_filtrar(caminho_csv, norte_min=1855050, norte_max=1855850,
                  este_min=2792250, este_max=2793250):
    df = pd.read_csv(caminho_csv)
    required_cols = ['northing', 'easting']
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"CSV is missing column: {col}")
    df = df[
        (df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
        (df['easting'] >= este_min) & (df['easting'] <= este_max)
    ]
    return df

# -------- 2. Carregar dados ASC/DESC --------
csv_asc = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
df_asc = ler_e_filtrar(csv_asc)
componente = "up"

# -------- 3. Melt das colunas de datas --------
def melt_asc(df, comp):
    static_cols = df.columns[:24]
    date_cols = df.columns[24:]
    df_melt = df.melt(id_vars=static_cols, value_vars=date_cols,
                      var_name='date', value_name=comp)
    df_melt['date'] = pd.to_datetime(df_melt['date'], format='%Y%m%d')
    return df_melt

df_melt = melt_asc(df_asc, componente)

# -------- 4. Pivot para matriz temporal --------
serie_matrix = df_melt.pivot_table(
    index=['easting','northing'],
    columns='date',
    values=componente
)
tempo = serie_matrix.columns

# -------- 5. Criar GeoDataFrame --------
gdf = gpd.GeoDataFrame(
    serie_matrix.reset_index()[['easting','northing']],
    geometry=gpd.points_from_xy(
        serie_matrix.reset_index().easting,
        serie_matrix.reset_index().northing
    ),
    crs="EPSG:3035"
)
gdf_web = gdf.to_crs(epsg=3857)

# -------- 6. KMeans clustering espacial --------
coords = np.vstack([gdf_web.geometry.x, gdf_web.geometry.y]).T
n_clusters = 4
kmeans = KMeans(n_clusters=n_clusters, random_state=0, n_init=10).fit(coords)
gdf_web["cluster"] = kmeans.labels_
serie_matrix["cluster"] = kmeans.labels_

# -------- 7. Escolher um cluster espacial --------
cluster_escolhido = 3
serie_matrix_reset = serie_matrix.reset_index()
subset_idx = serie_matrix_reset["cluster"] == cluster_escolhido

# Subset
subset_matrix = serie_matrix_reset[subset_idx].drop(columns="cluster")
gdf_web_reset = gdf_web.reset_index(drop=True)
subset_gdf = gdf_web_reset[subset_idx.values]

# -------- 8. Preencher NaNs --------
subset_matrix_filled = subset_matrix.copy()
subset_matrix_filled = subset_matrix_filled.reindex(columns=tempo)
subset_matrix_filled = subset_matrix_filled.fillna(method='ffill', axis=1).fillna(method='bfill', axis=1)

# -------- 9. KMeans temporal --------
X = subset_matrix_filled.values
X_scaled = StandardScaler().fit_transform(X)
n_clusters_temporal = 4
kmeans_temporal = KMeans(n_clusters=n_clusters_temporal, random_state=0, n_init=10).fit(X_scaled)
subset_matrix_filled["cluster_temporal"] = kmeans_temporal.labels_
subset_gdf["cluster_temporal"] = kmeans_temporal.labels_

# -------- 10. Carregar variáveis ambientais --------
df_nivel = pd.read_excel("data/alqueva_nivel.xlsx")
df_nivel['data'] = pd.to_datetime(df_nivel['data'])
df_nivel.set_index('data', inplace=True)

df_temp = pd.read_excel("data/alqueva_temp.xlsx")
df_temp['data'] = pd.to_datetime(df_temp['data'])
df_temp.set_index('data', inplace=True)

# -------- 11. Loop por sub-cluster (uma figura por sub-cluster) --------
for cluster_id in sorted(subset_matrix_filled["cluster_temporal"].unique()):
    idx = subset_matrix_filled["cluster_temporal"] == cluster_id
    series_cluster = subset_matrix_filled[idx].drop(columns="cluster_temporal").values
    media_cluster = np.nanmean(series_cluster, axis=0)
    
    datas = tempo
    up_values = media_cluster
    
    nivel_alinhado = df_nivel.reindex(datas, method='nearest')['nivel'].values
    temp_alinhado = df_temp.reindex(datas, method='nearest')['med'].values
    
    valid = ~np.isnan(up_values) & ~np.isnan(nivel_alinhado) & ~np.isnan(temp_alinhado)
    if valid.sum() < 3:
        continue
    
    up_clean = up_values[valid]
    nivel_clean = nivel_alinhado[valid]
    temp_clean = temp_alinhado[valid]
    datas_clean = datas[valid]
    
    corr_nivel = np.corrcoef(up_clean, nivel_clean)[0,1]
    corr_temp = np.corrcoef(up_clean, temp_clean)[0,1]
    
    # -------- Figura (uma por sub-cluster) --------
    fig = plt.figure(figsize=(28,18))
    gs = gridspec.GridSpec(
        2, 3,
        height_ratios=[1.2, 2],
        width_ratios=[6, 1.2, 1.2],
        hspace=0.35,
        wspace=0.35
    )
    
    # --- Série temporal com eixos secundários ---
    ax_series = fig.add_subplot(gs[0,:])
    line_up, = ax_series.plot(datas_clean, up_clean, color="black", linewidth=2, label="Deslocamento (mm)")
    ax_series.set_title(f"Série Média - Sub-cluster {cluster_id}", fontsize=24)
    ax_series.set_ylabel("Deslocamento (mm)", fontsize=18, color="black")
    ax_series.tick_params(axis='y', labelsize=16, labelcolor="black")
    ax_series.set_xlabel("Data", fontsize=18)
    ax_series.tick_params(axis='x', labelsize=16)
    
    ax_nivel = ax_series.twinx()
    line_nivel, = ax_nivel.plot(datas_clean, nivel_clean, color="blue", linewidth=2, label="Nível (m)")
    ax_nivel.set_ylabel("Nível (m)", color="blue", fontsize=18)
    ax_nivel.tick_params(axis='y', labelsize=16, labelcolor="blue")
    
    ax_temp = ax_series.twinx()
    ax_temp.spines["right"].set_position(("axes", 1.05))
    line_temp, = ax_temp.plot(datas_clean, temp_clean, color="red", linewidth=2, label="Temperatura (°C)")
    ax_temp.set_ylabel("Temperatura (°C)", color="red", fontsize=18)
    ax_temp.tick_params(axis='y', labelsize=16, labelcolor="red")
    
    ax_series.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=6, maxticks=10))
    ax_series.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    
    lines = [line_up, line_nivel, line_temp]
    labels = ["Deslocamento (mm)", "Nível (m)", "Temperatura (°C)"]
    ax_series.legend(lines, labels, loc="upper left", fontsize=16, frameon=False)
    
    # --- Mapa ---
    ax_map = fig.add_subplot(gs[1,0])
    gdf_web.plot(ax=ax_map, color="lightgray", markersize=20, alpha=0.5)
    subset_gdf[subset_gdf["cluster_temporal"]==cluster_id].plot(
        ax=ax_map, color="red", markersize=60, edgecolor="black"
    )
    ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
    ax_map.set_title(f"Pontos do Sub-cluster {cluster_id}", fontsize=20)
    ax_map.set_axis_off()
    
    # --- Correlação up vs nível ---
    ax_corr_nivel = fig.add_subplot(gs[1,1])
    sns.scatterplot(x=up_clean, y=nivel_clean, color="black", s=80, ax=ax_corr_nivel)
    sns.regplot(x=up_clean, y=nivel_clean, scatter=False, color="blue",
                line_kws={"lw":2,"ls":"--"}, ci=None, ax=ax_corr_nivel)
    ax_corr_nivel.set_title(f'up vs nível: r={corr_nivel:.2f}', fontsize=18)
    ax_corr_nivel.set_xlabel("Deslocamento (mm)", fontsize=16)
    ax_corr_nivel.set_ylabel("Nível (m)", fontsize=16)
    ax_corr_nivel.tick_params(axis='both', labelsize=14)
    
    # --- Correlação up vs temperatura ---
    ax_corr_temp = fig.add_subplot(gs[1,2])
    sns.scatterplot(x=up_clean, y=temp_clean, color="black", s=80, ax=ax_corr_temp)
    sns.regplot(x=up_clean, y=temp_clean, scatter=False, color="red",
                line_kws={"lw":2,"ls":"--"}, ci=None, ax=ax_corr_temp)
    ax_corr_temp.set_title(f'up vs temperatura: r={corr_temp:.2f}', fontsize=18)
    ax_corr_temp.set_xlabel("Deslocamento (mm)", fontsize=16)
    ax_corr_temp.set_ylabel("Temperatura (°C)", fontsize=16)
    ax_corr_temp.tick_params(axis='both', labelsize=14)
    
    plt.tight_layout()
    plt.show()



3. Correção da linha de base (baseline correction)

O que é:
- Ajustar as séries para remover deslocamentos artificiais ou tendências iniciais incorretas:
- Ex.: valores iniciais de cada ponto podem ter offset por processamento InSAR.
- Normalizar todas as séries para iniciar em zero ou na média de um período de referência.

Como usar:
- Subtrair o valor inicial ou média inicial da série.
- Pode ser feito antes de qualquer análise temporal ou clustering para reduzir dispersão artificial.

Benefício:
- Facilita comparação entre pontos, clusters e sub-clusters.
- Melhora interpretação da tendência real do deslocamento

ortho

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from matplotlib.cm import ScalarMappable
import contextily as ctx
import seaborn as sns
import matplotlib.gridspec as gridspec
from mpl_toolkits.axes_grid1 import make_axes_locatable

# -------- 1. Função para ler e filtrar CSV --------
def ler_e_filtrar(caminho_csv, norte_min=1855450, norte_max=1855650,
                  este_min=2792550, este_max=2792950):
    df = pd.read_csv(caminho_csv)
    required_cols = ['northing', 'easting']
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"CSV is missing column: {col}")
    df = df[
        (df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
        (df['easting'] >= este_min) & (df['easting'] <= este_max)
    ]
    return df

# -------- 2. Carregar dados ORTHO --------
csv_ortho = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
df_ortho = ler_e_filtrar(csv_ortho)
componente = "up"
vmin, vcenter, vmax = -20, 0, 20
usar_colormap_reverso = True

# -------- 3. Melt das colunas de datas --------
def melt_ortho(df, comp):
    static_cols = df.columns[:13]
    date_cols = df.columns[13:]
    df_melt = df.melt(id_vars=static_cols, value_vars=date_cols,
                      var_name='date', value_name=comp)
    df_melt['date'] = pd.to_datetime(df_melt['date'], format='%Y%m%d')
    return df_melt

df_melt = melt_ortho(df_ortho, componente)

# -------- 4. GeoDataFrame geral --------
df_media = df_melt.groupby(['easting','northing'], as_index=False).mean(numeric_only=True)
gdf_all = gpd.GeoDataFrame(
    df_media,
    geometry=gpd.points_from_xy(df_media['easting'], df_media['northing']),
    crs="EPSG:3035"
)
gdf_web = gdf_all.to_crs(3857)
xmin, ymin, xmax, ymax = gdf_web.total_bounds

# -------- 5. Colormap --------
cmap = plt.cm.jet
if usar_colormap_reverso:
    cmap = cmap.reversed()
norm = colors.TwoSlopeNorm(vmin=vmin, vcenter=vcenter, vmax=vmax)

# -------- 6. Dados ambientais --------
df_nivel = pd.read_excel("data/alqueva_nivel.xlsx")
df_nivel['data'] = pd.to_datetime(df_nivel['data'])
df_nivel.set_index('data', inplace=True)

df_temp = pd.read_excel("data/alqueva_temp.xlsx")
df_temp['data'] = pd.to_datetime(df_temp['data'])
df_temp.set_index('data', inplace=True)

# -------- 7. Pontos a analisar (todos) --------
pontos = df_melt[['easting','northing']].drop_duplicates()

# -------- 8. Loop por ponto para gerar figuras --------
tabela_resultados = []

for _, row in pontos.iterrows():
    ponto = (row['easting'], row['northing'])
    serie = df_melt[(df_melt['easting']==ponto[0]) & (df_melt['northing']==ponto[1])].sort_values('date')
    
    datas = serie['date']
    up_values = serie[componente]
    
    nivel_alinhado = df_nivel.reindex(datas, method='nearest')['nivel'].values
    temp_alinhado = df_temp.reindex(datas, method='nearest')['med'].values
    
    valid = ~np.isnan(up_values) & ~np.isnan(nivel_alinhado) & ~np.isnan(temp_alinhado)
    if valid.sum() < 3:
        continue
    
    up_clean = up_values[valid]
    nivel_clean = nivel_alinhado[valid]
    temp_clean = temp_alinhado[valid]
    datas_clean = datas[valid]
    
    corr_nivel = np.corrcoef(up_clean, nivel_clean)[0,1]
    corr_temp = np.corrcoef(up_clean, temp_clean)[0,1]
    
    tabela_resultados.append({
        'easting': ponto[0],
        'northing': ponto[1],
        'correlacao_nivel': corr_nivel,
        'correlacao_temp': corr_temp
    })
    
    # --- Figura ---
    fig = plt.figure(figsize=(20,12))
    gs = gridspec.GridSpec(3,2, height_ratios=[2,1.2,1.2], width_ratios=[1.2,1])
    
    # --- Mapa ---
    ax_map = fig.add_subplot(gs[0,0])
    divider = make_axes_locatable(ax_map)
    cax = divider.append_axes("right", size="5%", pad=0.05)
    
    gdf_web.plot(ax=ax_map, column=componente, cmap=cmap, norm=norm, markersize=5, legend=True, cax=cax)
    gpd.GeoDataFrame(geometry=gpd.points_from_xy([ponto[0]],[ponto[1]]), crs="EPSG:3035").to_crs(3857).plot(
        ax=ax_map, color='red', edgecolor='black', markersize=100
    )
    
    # --- Grelha 100x100 m ---
    step = 100
    for x in np.arange(xmin, xmax+step, step):
        ax_map.plot([x,x],[ymin,ymax], color='red', linewidth=0.6, alpha=0.7)
    for y in np.arange(ymin, ymax+step, step):
        ax_map.plot([xmin,xmax],[y,y], color='red', linewidth=0.6, alpha=0.7)
    
    ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
    ax_map.set_xlim(xmin, xmax)
    ax_map.set_ylim(ymin, ymax)
    ax_map.set_title(f"Ponto ({ponto[0]:.0f}, {ponto[1]:.0f})")
    ax_map.set_axis_off()
    
    # --- Série temporal ---
    ax_series = fig.add_subplot(gs[0,1])
    ax_series.plot(datas_clean, up_clean, color="blue", label="up (mm)")
    ax_series.axhline(0, color='gray', linestyle='--', linewidth=0.7)
    
    ax_nivel = ax_series.twinx()
    ax_nivel.plot(datas_clean, nivel_clean, color="green", label="nível (m)")
    ax_nivel.set_ylabel("Nível (m)", color="green")
    ax_nivel.tick_params(axis='y', labelcolor="green")
    
    ax_temp = ax_series.twinx()
    ax_temp.spines["right"].set_position(("axes", 1.1))
    ax_temp.plot(datas_clean, temp_clean, color="orange", label="temperatura (°C)")
    ax_temp.set_ylabel("Temp (°C)", color="orange")
    ax_temp.tick_params(axis='y', labelcolor="orange")
    
    ax_series.set_title("Série Temporal: up vs Nível vs Temperatura")
    ax_series.set_ylabel("Deslocamento (mm)")
    ax_series.set_xlabel("Data")
    ax_series.grid(True)
    
    # --- Dispersão up vs nível ---
    ax_corr_nivel = fig.add_subplot(gs[1,:])
    sns.scatterplot(x=up_clean, y=nivel_clean, color="blue", label="Pontos", ax=ax_corr_nivel)
    sns.regplot(x=up_clean, y=nivel_clean, scatter=False, color="red", line_kws={"lw":2,"ls":"--"}, ax=ax_corr_nivel)
    ax_corr_nivel.set_title(f'Correlação up vs nível: r={corr_nivel:.2f}')
    ax_corr_nivel.set_xlabel("Deslocamento (mm)")
    ax_corr_nivel.set_ylabel("Nível (m)")
    ax_corr_nivel.grid(True)
    
    # --- Dispersão up vs temperatura ---
    ax_corr_temp = fig.add_subplot(gs[2,:])
    sns.scatterplot(x=up_clean, y=temp_clean, color="purple", label="Pontos", ax=ax_corr_temp)
    sns.regplot(x=up_clean, y=temp_clean, scatter=False, color="orange", line_kws={"lw":2,"ls":"--"}, ax=ax_corr_temp)
    ax_corr_temp.set_title(f'Correlação up vs temperatura: r={corr_temp:.2f}')
    ax_corr_temp.set_xlabel("Deslocamento (mm)")
    ax_corr_temp.set_ylabel("Temperatura (°C)")
    ax_corr_temp.grid(True)
    
    plt.tight_layout()
    plt.show()


### Interpolador linear:
- O método "linear" aqui cria uma triangulação e interpola linearmente, o que é bom, mas exige que os pontos estejam bem distribuídos. Pode falhar ou dar NaN se o ponto estiver fora do fecho convexo dos dados.

In [ ]:
# Para cada data comum:
# Interpolar espacialmente os dados da órbita ASC para os pontos da órbita DESC (ou vice-versa).
# Depois combinar os dois conjuntos em cada ponto da malha.
# Finalmente, calcular a média dos componentes up, east, north.

# Interpolar usando scipy.interpolate.griddata() só nas datas que existem nas duas órbitas ASC e DESC.
# Sem interpolação temporal.
# Usar malha não regular: os pontos reais da outra órbita.

# Pontos = posições da órbita ascendente (mesmos easting e northing de df_asc);

import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from matplotlib.cm import ScalarMappable
from scipy.interpolate import griddata
import contextily as ctx

# --- PARÂMETROS EDITÁVEIS ---
csv_desc = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
csv_asc = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"

#componente = "up"     # 'up', 'east' ou 'north'
componente = "up" 
vmin = -5           # mínimo da escala (mm)
vcenter = 0          # centro da escala (normalmente zero)
vmax = 5            # máximo da escala (mm)
usar_colormap_reverso = True

# --- 1. Função para ler e filtrar ---
def ler_e_filtrar(caminho_csv):
    df = pd.read_csv(caminho_csv)
    # Filtro espacial (Alqueva)
    norte_min, norte_max = 1855050, 1855850
    este_min, este_max = 2792250, 2793250
    df = df[
        (df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
        (df['easting'] >= este_min) & (df['easting'] <= este_max)
    ]
    return df

df_asc = ler_e_filtrar(csv_asc)
df_desc = ler_e_filtrar(csv_desc)

# --- 2. Funções para melt e montar dataframe componentes ---
def melt(df, comp):
    static = ['pid', 'easting', 'northing']
    dates = [c for c in df.columns if c.startswith('20')]
    df_temp = df[static + dates].copy()
    df_melt = df_temp.melt(id_vars=static, var_name='date', value_name=comp)
    df_melt['date'] = pd.to_datetime(df_melt['date'], format='%Y%m%d')
    return df_melt

def get_componentes(df, prefix):
    up = melt(df, f'up_{prefix}')
    east = melt(df, f'east_{prefix}')
    north = melt(df, f'north_{prefix}')
    merged = up.merge(east, on=['pid', 'easting', 'northing', 'date'])
    merged = merged.merge(north, on=['pid', 'easting', 'northing', 'date'])
    return merged

asc = get_componentes(df_asc, 'asc')
desc = get_componentes(df_desc, 'desc')

# --- 3. Datas comuns ---
datas_comuns = np.intersect1d(asc['date'].unique(), desc['date'].unique())

# --- 4. Interpolação espacial por data ---
resultados = []

for data in datas_comuns:
    asc_data = asc[asc['date'] == data]
    desc_data = desc[desc['date'] == data]

    pontos_asc = asc_data[['easting', 'northing']].values
    pontos_desc = desc_data[['easting', 'northing']].values

    interp = {}
    for comp in ['up_asc', 'east_asc', 'north_asc']:
        interp[comp] = griddata(
            pontos_asc,
            asc_data[comp].values,
            pontos_desc,
            method='linear'
        )

    df_comb = pd.DataFrame({
        'easting': desc_data['easting'].values,
        'northing': desc_data['northing'].values,
        'date': data,
        'up': np.nanmean([interp['up_asc'], desc_data['up_desc'].values], axis=0),
        'east': np.nanmean([interp['east_asc'], desc_data['east_desc'].values], axis=0),
        'north': np.nanmean([interp['north_asc'], desc_data['north_desc'].values], axis=0),
    })

    resultados.append(df_comb)

df_final = pd.concat(resultados, ignore_index=True)
df_final.to_csv("alqueva_comb_espacial_linear.csv", index=False)

# --- 5. Calcular média por ponto ---
df_media = df_final.groupby(['easting', 'northing'], as_index=False).mean(numeric_only=True)

# --- 6. Criar GeoDataFrame e converter ---
gdf = gpd.GeoDataFrame(
    df_media,
    geometry=gpd.points_from_xy(df_media['easting'], df_media['northing']),
    crs="EPSG:3035"
)
gdf_web = gdf.to_crs(epsg=3857)

# --- 7. Configurar colormap e normalização ---
cmap = plt.cm.jet
if usar_colormap_reverso:
    cmap = cmap.reversed()

norma = colors.TwoSlopeNorm(vmin=vmin, vcenter=vcenter, vmax=vmax)

# --- 8. Plotar ---
fig, ax = plt.subplots(figsize=(9, 6))
gdf_web.plot(
    ax=ax,
    column=componente,
    cmap=cmap,
    markersize=10,
    norm=norma,
    legend=False
)


# Limites do mapa
xmin, ymin, xmax, ymax = gdf_web.total_bounds

# Passo da grelha (100 m)
step = 100  

# Linhas verticais
for x in np.arange(xmin, xmax + step, step):
    ax.plot([x, x], [ymin, ymax], color='red', linewidth=0.8, alpha=0.8)

# Linhas horizontais
for y in np.arange(ymin, ymax + step, step):
    ax.plot([xmin, xmax], [y, y], color='red', linewidth=0.8, alpha=0.8)



ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)
ax.set_axis_off()
plt.title(f"Deslocamento médio ({componente}) - ASC + DESC (linear)", fontsize=16)

sm = ScalarMappable(cmap=cmap, norm=norma)
sm._A = []
cbar = plt.colorbar(sm, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label("Deslocamento (mm)", fontsize=14)

plt.tight_layout()
plt.show()


In [ ]:
# Para cada data comum:
# Interpolar espacialmente os dados da órbita ASC para os pontos da órbita DESC (ou vice-versa).
# Depois combinar os dois conjuntos em cada ponto da malha.
# Finalmente, calcular a média dos componentes up, east, north.

# Interpolar usando scipy.interpolate.griddata() só nas datas que existem nas duas órbitas ASC e DESC.
# Sem interpolação temporal.
# Usar malha não regular: os pontos reais da outra órbita.

# Pontos = posições da órbita ascendente (mesmos easting e northing de df_asc);

import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from matplotlib.cm import ScalarMappable
from scipy.interpolate import griddata
import contextily as ctx

# --- PARÂMETROS EDITÁVEIS ---
csv_desc = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
csv_asc = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"

#componente = "up"     # 'up', 'east' ou 'north'
componente = "up" 
vmin = -5           # mínimo da escala (mm)
vcenter = 0          # centro da escala (normalmente zero)
vmax = 5            # máximo da escala (mm)
usar_colormap_reverso = True

# --- 1. Função para ler e filtrar ---
def ler_e_filtrar(caminho_csv):
    df = pd.read_csv(caminho_csv)
    # Filtro espacial (Alqueva)
    norte_min, norte_max = 1855050, 1855850
    este_min, este_max = 2792250, 2793250
    df = df[
        (df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
        (df['easting'] >= este_min) & (df['easting'] <= este_max)
    ]
    return df

df_asc = ler_e_filtrar(csv_asc)
df_desc = ler_e_filtrar(csv_desc)

# --- 2. Funções para melt e montar dataframe componentes ---
def melt(df, comp):
    static = ['pid', 'easting', 'northing']
    dates = [c for c in df.columns if c.startswith('20')]
    df_temp = df[static + dates].copy()
    df_melt = df_temp.melt(id_vars=static, var_name='date', value_name=comp)
    df_melt['date'] = pd.to_datetime(df_melt['date'], format='%Y%m%d')
    return df_melt

def get_componentes(df, prefix):
    up = melt(df, f'up_{prefix}')
    east = melt(df, f'east_{prefix}')
    north = melt(df, f'north_{prefix}')
    merged = up.merge(east, on=['pid', 'easting', 'northing', 'date'])
    merged = merged.merge(north, on=['pid', 'easting', 'northing', 'date'])
    return merged

asc = get_componentes(df_asc, 'asc')
desc = get_componentes(df_desc, 'desc')

# --- 3. Datas comuns ---
datas_comuns = np.intersect1d(asc['date'].unique(), desc['date'].unique())

# --- 4. Interpolação espacial por data ---
resultados = []

for data in datas_comuns:
    asc_data = asc[asc['date'] == data]
    desc_data = desc[desc['date'] == data]

    pontos_asc = asc_data[['easting', 'northing']].values
    pontos_desc = desc_data[['easting', 'northing']].values

    interp = {}
    for comp in ['up_asc', 'east_asc', 'north_asc']:
        interp[comp] = griddata(
            pontos_asc,
            asc_data[comp].values,
            pontos_desc,
            method='linear'
        )

    df_comb = pd.DataFrame({
        'easting': desc_data['easting'].values,
        'northing': desc_data['northing'].values,
        'date': data,
        'up': np.nanmean([interp['up_asc'], desc_data['up_desc'].values], axis=0),
        'east': np.nanmean([interp['east_asc'], desc_data['east_desc'].values], axis=0),
        'north': np.nanmean([interp['north_asc'], desc_data['north_desc'].values], axis=0),
    })

    resultados.append(df_comb)

df_final = pd.concat(resultados, ignore_index=True)
df_final.to_csv("alqueva_comb_espacial_linear.csv", index=False)

# --- 5. Calcular média por ponto ---
df_media = df_final.groupby(['easting', 'northing'], as_index=False).mean(numeric_only=True)

# --- 6. Criar GeoDataFrame e converter ---
gdf = gpd.GeoDataFrame(
    df_media,
    geometry=gpd.points_from_xy(df_media['easting'], df_media['northing']),
    crs="EPSG:3035"
)
gdf_web = gdf.to_crs(epsg=3857)

# --- 7. Configurar colormap e normalização ---
cmap = plt.cm.jet
if usar_colormap_reverso:
    cmap = cmap.reversed()

norma = colors.TwoSlopeNorm(vmin=vmin, vcenter=vcenter, vmax=vmax)

# --- 8. Plotar ---
fig, ax = plt.subplots(figsize=(9, 6))
gdf_web.plot(
    ax=ax,
    column=componente,
    cmap=cmap,
    markersize=10,
    norm=norma,
    legend=False
)


# Limites do mapa
xmin, ymin, xmax, ymax = gdf_web.total_bounds

# Passo da grelha (100 m)
step = 100  

# Linhas verticais
for x in np.arange(xmin, xmax + step, step):
    ax.plot([x, x], [ymin, ymax], color='red', linewidth=0.8, alpha=0.8)

# Linhas horizontais
for y in np.arange(ymin, ymax + step, step):
    ax.plot([xmin, xmax], [y, y], color='red', linewidth=0.8, alpha=0.8)



ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)
ax.set_axis_off()
plt.title(f"Deslocamento médio ({componente}) - ASC + DESC (linear)", fontsize=16)

sm = ScalarMappable(cmap=cmap, norm=norma)
sm._A = []
cbar = plt.colorbar(sm, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label("Deslocamento (mm)", fontsize=14)

plt.tight_layout()
plt.show()

### Ponto mais instável em termos de deslocamento:
- Neste último bloco de código, o ponto está a ser escolhido com base na variabilidade (instabilidade), ou seja, no desvio padrão do deslocamento vertical ("up"), não com base na correlação com o nível da albufeira.

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from matplotlib.cm import ScalarMappable
from scipy.interpolate import griddata
import contextily as ctx
import seaborn as sns
import matplotlib.gridspec as gridspec
from mpl_toolkits.axes_grid1 import make_axes_locatable

# ----------- Parâmetros editáveis ------------
#componente = "up"
componente = "up"
vmin, vcenter, vmax = -5, 0, 5  # mm
top_n = 1  # Número de pontos mais instáveis
usar_colormap_reverso = True

# ----------- Leitura dados ---------
#df_asc = pd.read_csv("...")  # caminho ASC
#df_desc = pd.read_csv("...")  # caminho DESC
df_nivel = pd.read_excel("data/alqueva_nivel.xlsx")
df_nivel['data'] = pd.to_datetime(df_nivel['data'])
df_nivel.set_index('data', inplace=True)

# ----------- Filtro espacial ---------
def filtro_alqueva(df):
    return df[
        (df['northing'] >= 1855050) & (df['northing'] <= 1855850) &
        (df['easting'] >= 2792250) & (df['easting'] <= 2793250)
    ]
df_asc = filtro_alqueva(df_asc)
df_desc = filtro_alqueva(df_desc)

# ----------- Pré-processamento ------------
def melt(df, comp):
    static = ['pid', 'easting', 'northing']
    datas = [c for c in df.columns if c.startswith('20')]
    df_temp = df[static + datas].copy()
    df_melt = df_temp.melt(id_vars=static, var_name='date', value_name=comp)
    df_melt['date'] = pd.to_datetime(df_melt['date'], format='%Y%m%d')
    return df_melt

def get_componentes(df, prefix):
    up = melt(df, f'up_{prefix}')
    east = melt(df, f'east_{prefix}')
    north = melt(df, f'north_{prefix}')
    merged = up.merge(east, on=['pid', 'easting', 'northing', 'date'])
    merged = merged.merge(north, on=['pid', 'easting', 'northing', 'date'])
    return merged

asc = get_componentes(df_asc, 'asc')
desc = get_componentes(df_desc, 'desc')

datas_comuns = np.intersect1d(asc['date'].unique(), desc['date'].unique())

# ----------- Interpolação espacial e média ------------
resultados = []
for data in datas_comuns:
    asc_data = asc[asc['date'] == data]
    desc_data = desc[desc['date'] == data]
    pontos_asc = asc_data[['easting', 'northing']].values
    pontos_desc = desc_data[['easting', 'northing']].values

    interp = {
        comp: griddata(pontos_asc, asc_data[comp].values, pontos_desc, method='linear')
        for comp in ['up_asc', 'east_asc', 'north_asc']
    }

    df_comb = pd.DataFrame({
        'easting': desc_data['easting'].values,
        'northing': desc_data['northing'].values,
        'date': data,
        'up': np.nanmean([interp['up_asc'], desc_data['up_desc'].values], axis=0),
        'east': np.nanmean([interp['east_asc'], desc_data['east_desc'].values], axis=0),
        'north': np.nanmean([interp['north_asc'], desc_data['north_desc'].values], axis=0),
    })

    resultados.append(df_comb)

df_final = pd.concat(resultados, ignore_index=True)

# ----------- Média por ponto ------------
df_mean = df_final.groupby(['easting', 'northing'], as_index=False).mean(numeric_only=True)
df_mean['std_up'] = df_final.groupby(['easting', 'northing'])['up'].std().values  # Desvio padrão

# ----------- Top N pontos mais instáveis ------------
df_top = df_mean.nlargest(top_n, 'std_up')

# ----------- GeoDataFrame ------------
gdf_all = gpd.GeoDataFrame(df_mean, geometry=gpd.points_from_xy(df_mean.easting, df_mean.northing), crs="EPSG:3035").to_crs(epsg=3857)
gdf_top = gpd.GeoDataFrame(df_top, geometry=gpd.points_from_xy(df_top.easting, df_top.northing), crs="EPSG:3035").to_crs(epsg=3857)

# ----------- Normalização para mapa ------------
norm = colors.TwoSlopeNorm(vmin=vmin, vcenter=vcenter, vmax=vmax)
xmin, ymin, xmax, ymax = gdf_all.total_bounds
cmap = plt.cm.jet.reversed() if usar_colormap_reverso else plt.cm.jet

# ----------- Loop pelos top N ------------
for _, row in df_top.iterrows():
    ponto = (row['easting'], row['northing'])
    serie = df_final[(df_final['easting'] == ponto[0]) & (df_final['northing'] == ponto[1])]
    serie = serie.sort_values('date')
    datas = serie['date']
    up_values = serie['up']

    # Alinhar com nível
    nivel_alinhado = df_nivel.reindex(datas, method='nearest')['nivel'].values
    valid = ~np.isnan(up_values) & ~np.isnan(nivel_alinhado)
    up_clean = up_values[valid]
    nivel_clean = nivel_alinhado[valid]
    datas_clean = datas[valid]

    # Correlação e tendência
    corr_value = np.corrcoef(up_clean, nivel_clean)[0,1]
    coef = np.polyfit(datas_clean.map(pd.Timestamp.toordinal), up_clean, deg=1)  # tendência

    # ----------- Plot melhorado com gridspec ------------
    fig = plt.figure(figsize=(18, 10))
    gs = gridspec.GridSpec(2, 2, height_ratios=[2, 1], width_ratios=[1.3, 1])

    # --- Mapa ---
    ax_map = fig.add_subplot(gs[0, 0])
    divider = make_axes_locatable(ax_map)
    cax = divider.append_axes("right", size="5%", pad=0.05)

    gdf_all.plot(ax=ax_map, column='up', cmap=cmap, norm=norm, markersize=5, legend=True, cax=cax)
    gpd.GeoDataFrame(
    geometry=gpd.points_from_xy([row['easting']], [row['northing']]), 
    crs="EPSG:3035"
        ).to_crs(3857).plot(ax=ax_map, color='red', edgecolor='black', markersize=100)
    ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
    ax_map.set_xlim(xmin, xmax)
    ax_map.set_ylim(ymin, ymax)
    ax_map.set_title(f"Ponto mais instável | Média: {row['up']:.2f} mm")
    ax_map.set_axis_off()

    # --- Série temporal ---
    ax_series = fig.add_subplot(gs[0, 1])
    ax_series.plot(datas, up_values, color="blue", label="up")
    ax_series.axhline(0, color='gray', linestyle='--', linewidth=0.7)
    ax_series.set_title("Série Temporal: up vs Nível")
    ax_series.set_ylabel("Deslocamento (mm)")
    ax_series.set_xlabel("Data")
    ax_series.grid(True)
    if len(nivel_clean) > 0:
        ax2 = ax_series.twinx()
        ax2.plot(datas, nivel_alinhado, color="green", label="Nível")
        ax2.set_ylabel("Nível (m)", color="green")
        ax2.tick_params(axis='y', labelcolor="green")
        ax_series.legend(loc='upper left')
        ax2.legend(loc='upper right')

    # --- Dispersão + regressão ---
    ax_corr = fig.add_subplot(gs[1, :])
    sns.scatterplot(x=up_clean, y=nivel_clean, color="blue", label="Pontos", ax=ax_corr)
    sns.regplot(x=up_clean, y=nivel_clean, scatter=False, color="red", line_kws={"lw": 2, "ls": "--"}, ax=ax_corr)
    ax_corr.set_title(f'Correlação up vs Nível = {corr_value:.2f} | Tendência = {coef[0]:.4f} mm/dia')
    ax_corr.set_xlabel("Deslocamento (mm)")
    ax_corr.set_ylabel("Nível (m)")
    ax_corr.grid(True)

    plt.tight_layout()
    plt.show()

#### os pontos estão a ser escolhidos com base na maior correlação (em valor absoluto) entre o deslocamento vertical up e os fatores ambientais (nível da água e temperatura) — não com base no maior deslocamento ou variabilidade diretamente.

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.gridspec as gridspec
from mpl_toolkits.axes_grid1 import make_axes_locatable
import contextily as ctx

# -------- Parâmetros --------
top_n = 1  # Quantos pontos mais instáveis considerar (pode usar todo conjunto também)
usar_colormap_reverso = True

# -------- Leitura dos dados ambientais --------
df_nivel = pd.read_excel("data/alqueva_nivel.xlsx")
df_nivel['data'] = pd.to_datetime(df_nivel['data'])
df_nivel.set_index('data', inplace=True)

df_temp = pd.read_excel("data/alqueva_temp.xlsx")
df_temp['data'] = pd.to_datetime(df_temp['data'])
df_temp.set_index('data', inplace=True)

# -------- Leitura dos deslocamentos (df_final) --------
# df_final deve ter as colunas: ['easting', 'northing', 'date', 'up']
# date em datetime, up em mm (deslocamento vertical)
# Exemplo de carregamento: df_final = pd.read_csv("seu_arquivo_de_up.csv")
# Aqui assumo que você já tem df_final pronto (gerado pelo seu primeiro script)
# Se quiser, posso ajudar a montar o carregamento + preprocessamento também

# -------- Seleção dos pontos mais instáveis (top_n) --------
variabilidade = df_final.groupby(['easting', 'northing'])['up'].std().reset_index()
variabilidade = variabilidade.rename(columns={'up': 'std_up'})
df_top = variabilidade.sort_values('std_up', ascending=False).head(top_n)

# -------- Calcula correlações para cada ponto --------
tabela_resultados = []

for idx, row in df_top.iterrows():
    ponto = (row['easting'], row['northing'])
    serie = df_final[
        (df_final['easting'] == ponto[0]) & (df_final['northing'] == ponto[1])
    ].sort_values('date')

    datas = serie['date']
    up_values = serie['up']

    nivel_alinhado = df_nivel.reindex(datas, method='nearest')['nivel'].values
    temp_alinhado = df_temp.reindex(datas, method='nearest')['med'].values  # ou 'max'/'min'

    valid = ~np.isnan(up_values) & ~np.isnan(nivel_alinhado) & ~np.isnan(temp_alinhado)

    if valid.sum() < 3:
        # Não tem dados suficientes para correlação
        continue

    up_clean = up_values[valid]
    nivel_clean = nivel_alinhado[valid]
    temp_clean = temp_alinhado[valid]

    corr_nivel = np.corrcoef(up_clean, nivel_clean)[0, 1]
    corr_temp = np.corrcoef(up_clean, temp_clean)[0, 1]

    tabela_resultados.append({
        'easting': ponto[0],
        'northing': ponto[1],
        'correlacao_nivel': corr_nivel,
        'correlacao_temp': corr_temp
    })

# DataFrame final com correlações
df_resultados = pd.DataFrame(tabela_resultados)

# Seleciona o ponto com maior correlação absoluta com nível e temperatura
ponto_nivel = df_resultados.loc[df_resultados['correlacao_nivel'].abs().idxmax()]
ponto_temp = df_resultados.loc[df_resultados['correlacao_temp'].abs().idxmax()]

# Combina sem repetir pontos iguais
pontos_escolhidos = pd.DataFrame([ponto_nivel, ponto_temp]).drop_duplicates(subset=['easting', 'northing'])

print("Pontos escolhidos para análise detalhada:")
print(pontos_escolhidos)

# -------- Plot dos pontos selecionados --------
for _, row in pontos_escolhidos.iterrows():
    ponto = (row['easting'], row['northing'])
    serie = df_final[
        (df_final['easting'] == ponto[0]) & (df_final['northing'] == ponto[1])
    ].sort_values('date')

    datas = serie['date']
    up_values = serie['up']

    nivel_alinhado = df_nivel.reindex(datas, method='nearest')['nivel'].values
    temp_alinhado = df_temp.reindex(datas, method='nearest')['med'].values

    valid = ~np.isnan(up_values) & ~np.isnan(nivel_alinhado) & ~np.isnan(temp_alinhado)
    if valid.sum() < 3:
        continue

    up_clean = up_values[valid]
    nivel_clean = nivel_alinhado[valid]
    temp_clean = temp_alinhado[valid]
    datas_clean = datas[valid]

    corr_nivel = np.corrcoef(up_clean, nivel_clean)[0, 1]
    corr_temp = np.corrcoef(up_clean, temp_clean)[0, 1]

    fig = plt.figure(figsize=(20, 12))
    gs = gridspec.GridSpec(3, 2, height_ratios=[2, 1.2, 1.2], width_ratios=[1.2, 1])

    # --- Mapa ---
    ax_map = fig.add_subplot(gs[0, 0])
    divider = make_axes_locatable(ax_map)
    cax = divider.append_axes("right", size="5%", pad=0.05)

    gdf_all.plot(ax=ax_map, column='up', cmap=cmap, norm=norm, markersize=5, legend=True, cax=cax)
    gpd.GeoDataFrame(
        geometry=gpd.points_from_xy([ponto[0]], [ponto[1]]), crs="EPSG:3035"
    ).to_crs(3857).plot(ax=ax_map, color='red', edgecolor='black', markersize=100)
    ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
    ax_map.set_xlim(xmin, xmax)
    ax_map.set_ylim(ymin, ymax)
    ax_map.set_title(f"Ponto ({ponto[0]:.0f}, {ponto[1]:.0f})")
    ax_map.set_axis_off()

    # --- Série temporal ---
    ax_series = fig.add_subplot(gs[0, 1])
    ax_series.plot(datas_clean, up_clean, color="blue", label="up (mm)")
    ax_series.axhline(0, color='gray', linestyle='--', linewidth=0.7)

    ax_nivel = ax_series.twinx()
    ax_nivel.plot(datas_clean, nivel_clean, color="green", label="nível (m)")
    ax_nivel.set_ylabel("Nível (m)", color="green")
    ax_nivel.tick_params(axis='y', labelcolor="green")

    ax_temp = ax_series.twinx()
    ax_temp.spines["right"].set_position(("axes", 1.1))  # desloca eixo à direita
    ax_temp.plot(datas_clean, temp_clean, color="orange", label="temperatura (°C)")
    ax_temp.set_ylabel("Temp (°C)", color="orange")
    ax_temp.tick_params(axis='y', labelcolor="orange")

    ax_series.set_title("Série Temporal: up vs Nível vs Temperatura")
    ax_series.set_ylabel("Deslocamento (mm)")
    ax_series.set_xlabel("Data")
    ax_series.grid(True)

    # --- Dispersão up vs nível ---
    ax_corr_nivel = fig.add_subplot(gs[1, :])
    sns.scatterplot(x=up_clean, y=nivel_clean, color="blue", label="Pontos", ax=ax_corr_nivel)
    sns.regplot(x=up_clean, y=nivel_clean, scatter=False, color="red", line_kws={"lw": 2, "ls": "--"}, ax=ax_corr_nivel)
    ax_corr_nivel.set_title(f'Correlação up vs nível: r = {corr_nivel:.2f}')
    ax_corr_nivel.set_xlabel("Deslocamento (mm)")
    ax_corr_nivel.set_ylabel("Nível (m)")
    ax_corr_nivel.grid(True)

    # --- Dispersão up vs temperatura ---
    ax_corr_temp = fig.add_subplot(gs[2, :])
    sns.scatterplot(x=up_clean, y=temp_clean, color="purple", label="Pontos", ax=ax_corr_temp)
    sns.regplot(x=up_clean, y=temp_clean, scatter=False, color="orange", line_kws={"lw": 2, "ls": "--"}, ax=ax_corr_temp)
    ax_corr_temp.set_title(f'Correlação up vs temperatura: r = {corr_temp:.2f}')
    ax_corr_temp.set_xlabel("Deslocamento (mm)")
    ax_corr_temp.set_ylabel("Temperatura (°C)")
    ax_corr_temp.grid(True)

    plt.tight_layout()
    plt.show()

# ----- Gerar DataFrame com os resultados -----
df_resultados = pd.DataFrame(tabela_resultados)

# Ponto com maior correlação com o nível (em valor absoluto)
ponto_nivel = df_resultados.loc[df_resultados['correlacao_nivel'].abs().idxmax()]
# Ponto com maior correlação com a temperatura (em valor absoluto)
ponto_temp = df_resultados.loc[df_resultados['correlacao_temp'].abs().idxmax()]

# Combinar os dois (evitar repetição se forem o mesmo ponto)
pontos_escolhidos = pd.DataFrame([ponto_nivel, ponto_temp]).drop_duplicates(subset=['easting', 'northing'])

plt.tight_layout()
plt.show()


### correlação pontos com variáveis (nível, temperatura, ...)

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from matplotlib.cm import ScalarMappable
from scipy.interpolate import griddata
import contextily as ctx
import seaborn as sns
import matplotlib.gridspec as gridspec
from mpl_toolkits.axes_grid1 import make_axes_locatable

# ----------- Parâmetros editáveis ------------
componente = "up"
vmin, vcenter, vmax = -5, 0, 5  # mm
top_n = 1  # Número de pontos mais instáveis
usar_colormap_reverso = True

# ----------- Leitura dados ---------
#df_asc = pd.read_csv("...")  # caminho ASC
#df_desc = pd.read_csv("...")  # caminho DESC
df_nivel = pd.read_excel("data/alqueva_nivel.xlsx")
df_nivel['data'] = pd.to_datetime(df_nivel['data'])
df_nivel.set_index('data', inplace=True)

df_temp = pd.read_excel("data/alqueva_temp.xlsx")
df_temp['data'] = pd.to_datetime(df_temp['data'])
df_temp.set_index('data', inplace=True)

# ----------- Filtro espacial ---------
def filtro_alqueva(df):
    return df[
        (df['northing'] >= 1855050) & (df['northing'] <= 1855850) &
        (df['easting'] >= 2792250) & (df['easting'] <= 2793250)
    ]
df_asc = filtro_alqueva(df_asc)
df_desc = filtro_alqueva(df_desc)

# ----------- Pré-processamento ------------
def melt(df, comp):
    static = ['pid', 'easting', 'northing']
    datas = [c for c in df.columns if c.startswith('20')]
    df_temp = df[static + datas].copy()
    df_melt = df_temp.melt(id_vars=static, var_name='date', value_name=comp)
    df_melt['date'] = pd.to_datetime(df_melt['date'], format='%Y%m%d')
    return df_melt

def get_componentes(df, prefix):
    up = melt(df, f'up_{prefix}')
    east = melt(df, f'east_{prefix}')
    north = melt(df, f'north_{prefix}')
    merged = up.merge(east, on=['pid', 'easting', 'northing', 'date'])
    merged = merged.merge(north, on=['pid', 'easting', 'northing', 'date'])
    return merged

asc = get_componentes(df_asc, 'asc')
desc = get_componentes(df_desc, 'desc')

datas_comuns = np.intersect1d(asc['date'].unique(), desc['date'].unique())

# ----------- Interpolação espacial e média ------------
# Lista para guardar os resultados
tabela_resultados = []

for _, row in df_top.iterrows():  # <-- Correção aqui
    ponto = (row['easting'], row['northing'])
    serie = df_final[
        (df_final['easting'] == ponto[0]) & (df_final['northing'] == ponto[1])
    ].sort_values('date')

    datas = serie['date']
    up_values = serie['up']

    nivel_alinhado = df_nivel.reindex(datas, method='nearest')['nivel'].values
    temp_alinhado = df_temp.reindex(datas, method='nearest')['med'].values  # ou 'max'/'min'

    valid = ~np.isnan(up_values) & ~np.isnan(nivel_alinhado) & ~np.isnan(temp_alinhado)

    if valid.sum() < 3:
        continue

    up_clean = up_values[valid]
    nivel_clean = nivel_alinhado[valid]
    temp_clean = temp_alinhado[valid]
    datas_clean = datas[valid]

    corr_nivel = np.corrcoef(up_clean, nivel_clean)[0, 1]
    corr_temp = np.corrcoef(up_clean, temp_clean)[0, 1]

    # Salvar resultado
    tabela_resultados.append({
        'easting': ponto[0],
        'northing': ponto[1],
        'correlacao_nivel': corr_nivel,
        'correlacao_temp': corr_temp
    })

    # Plot
    fig = plt.figure(figsize=(20, 12))
    gs = gridspec.GridSpec(3, 2, height_ratios=[2, 1.2, 1.2], width_ratios=[1.2, 1])

    # --- Mapa ---
    ax_map = fig.add_subplot(gs[0, 0])
    divider = make_axes_locatable(ax_map)
    cax = divider.append_axes("right", size="5%", pad=0.05)

    gdf_all.plot(ax=ax_map, column='up', cmap=cmap, norm=norm, markersize=5, legend=True, cax=cax)
    gpd.GeoDataFrame(
        geometry=gpd.points_from_xy([ponto[0]], [ponto[1]]), crs="EPSG:3035"
    ).to_crs(3857).plot(ax=ax_map, color='red', edgecolor='black', markersize=100)
    ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
    ax_map.set_xlim(xmin, xmax)
    ax_map.set_ylim(ymin, ymax)
    ax_map.set_title(f"Ponto ({ponto[0]:.0f}, {ponto[1]:.0f})")
    ax_map.set_axis_off()

    # --- Série temporal ---
    ax_series = fig.add_subplot(gs[0, 1])
    ax_series.plot(datas_clean, up_clean, color="blue", label="up (mm)")
    ax_series.axhline(0, color='gray', linestyle='--', linewidth=0.7)

    ax_nivel = ax_series.twinx()
    ax_nivel.plot(datas_clean, nivel_clean, color="green", label="nível (m)")
    ax_nivel.set_ylabel("Nível (m)", color="green")
    ax_nivel.tick_params(axis='y', labelcolor="green")

    ax_temp = ax_series.twinx()
    ax_temp.spines["right"].set_position(("axes", 1.1))  # desloca eixo à direita
    ax_temp.plot(datas_clean, temp_clean, color="orange", label="temperatura (°C)")
    ax_temp.set_ylabel("Temp (°C)", color="orange")
    ax_temp.tick_params(axis='y', labelcolor="orange")

    ax_series.set_title("Série Temporal: up vs Nível vs Temperatura")
    ax_series.set_ylabel("Deslocamento (mm)")
    ax_series.set_xlabel("Data")
    ax_series.grid(True)

    # --- Dispersão up vs nível ---
    ax_corr_nivel = fig.add_subplot(gs[1, :])
    sns.scatterplot(x=up_clean, y=nivel_clean, color="blue", label="Pontos", ax=ax_corr_nivel)
    sns.regplot(x=up_clean, y=nivel_clean, scatter=False, color="red", line_kws={"lw": 2, "ls": "--"}, ax=ax_corr_nivel)
    ax_corr_nivel.set_title(f'Correlação up vs nível: r = {corr_nivel:.2f}')
    ax_corr_nivel.set_xlabel("Deslocamento (mm)")
    ax_corr_nivel.set_ylabel("Nível (m)")
    ax_corr_nivel.grid(True)

    # --- Dispersão up vs temperatura ---
    ax_corr_temp = fig.add_subplot(gs[2, :])
    sns.scatterplot(x=up_clean, y=temp_clean, color="purple", label="Pontos", ax=ax_corr_temp)
    sns.regplot(x=up_clean, y=temp_clean, scatter=False, color="orange", line_kws={"lw": 2, "ls": "--"}, ax=ax_corr_temp)
    ax_corr_temp.set_title(f'Correlação up vs temperatura: r = {corr_temp:.2f}')
    ax_corr_temp.set_xlabel("Deslocamento (mm)")
    ax_corr_temp.set_ylabel("Temperatura (°C)")
    ax_corr_temp.grid(True)

    plt.tight_layout()
    plt.show()

# ----- Gerar DataFrame com os resultados -----
df_resultados = pd.DataFrame(tabela_resultados)

# Ponto com maior correlação com o nível (em valor absoluto)
ponto_nivel = df_resultados.loc[df_resultados['correlacao_nivel'].abs().idxmax()]
# Ponto com maior correlação com a temperatura (em valor absoluto)
ponto_temp = df_resultados.loc[df_resultados['correlacao_temp'].abs().idxmax()]

# Combinar os dois (evitar repetição se forem o mesmo ponto)
pontos_escolhidos = pd.DataFrame([ponto_nivel, ponto_temp]).drop_duplicates(subset=['easting', 'northing'])

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from matplotlib.cm import ScalarMappable
from scipy.interpolate import griddata
import contextily as ctx
import seaborn as sns
import matplotlib.gridspec as gridspec
from mpl_toolkits.axes_grid1 import make_axes_locatable

# ----------- Parâmetros editáveis ------------
componente = "up"
vmin, vcenter, vmax = -5, 0, 5  # mm
top_n = 4  # Número de pontos mais instáveis
usar_colormap_reverso = True

# ----------- Leitura dados ---------
#df_asc = pd.read_csv("...")  # caminho ASC
#df_desc = pd.read_csv("...")  # caminho DESC
df_nivel = pd.read_excel("data/alqueva_nivel.xlsx")
df_nivel['data'] = pd.to_datetime(df_nivel['data'])
df_nivel.set_index('data', inplace=True)

df_temp = pd.read_excel("data/alqueva_temp.xlsx")
df_temp['data'] = pd.to_datetime(df_temp['data'])
df_temp.set_index('data', inplace=True)

# ----------- Filtro espacial ---------
def filtro_alqueva(df):
    return df[
        (df['northing'] >= 1855050) & (df['northing'] <= 1855850) &
        (df['easting'] >= 2792250) & (df['easting'] <= 2793250)
    ]
df_asc = filtro_alqueva(df_asc)
df_desc = filtro_alqueva(df_desc)

# ----------- Pré-processamento ------------
def melt(df, comp):
    static = ['pid', 'easting', 'northing']
    datas = [c for c in df.columns if c.startswith('20')]
    df_temp = df[static + datas].copy()
    df_melt = df_temp.melt(id_vars=static, var_name='date', value_name=comp)
    df_melt['date'] = pd.to_datetime(df_melt['date'], format='%Y%m%d')
    return df_melt

def get_componentes(df, prefix):
    up = melt(df, f'up_{prefix}')
    east = melt(df, f'east_{prefix}')
    north = melt(df, f'north_{prefix}')
    merged = up.merge(east, on=['pid', 'easting', 'northing', 'date'])
    merged = merged.merge(north, on=['pid', 'easting', 'northing', 'date'])
    return merged

asc = get_componentes(df_asc, 'asc')
desc = get_componentes(df_desc, 'desc')

datas_comuns = np.intersect1d(asc['date'].unique(), desc['date'].unique())

# ----------- Interpolação espacial e média ------------
# Lista para guardar os resultados
tabela_resultados = []

for _, row in df_top.iterrows():
    ponto = (row['easting'], row['northing'])
    serie = df_final[
        (df_final['easting'] == ponto[0]) & (df_final['northing'] == ponto[1])
    ].sort_values('date')

    datas = serie['date']
    up_values = serie['up']

    nivel_alinhado = df_nivel.reindex(datas, method='nearest')['nivel'].values
    temp_alinhado = df_temp.reindex(datas, method='nearest')['med'].values  # ou 'max'/'min'

    valid = ~np.isnan(up_values) & ~np.isnan(nivel_alinhado) & ~np.isnan(temp_alinhado)

    if valid.sum() < 3:
        continue

    up_clean = up_values[valid]
    nivel_clean = nivel_alinhado[valid]
    temp_clean = temp_alinhado[valid]
    datas_clean = datas[valid]

    corr_nivel = np.corrcoef(up_clean, nivel_clean)[0, 1]
    corr_temp = np.corrcoef(up_clean, temp_clean)[0, 1]

    # Salvar resultado
    tabela_resultados.append({
        'easting': ponto[0],
        'northing': ponto[1],
        'correlacao_nivel': corr_nivel,
        'correlacao_temp': corr_temp
    })

    # Plot
    fig = plt.figure(figsize=(20, 12))
    gs = gridspec.GridSpec(3, 2, height_ratios=[2, 1.2, 1.2], width_ratios=[1.2, 1])

    # --- Mapa ---
    ax_map = fig.add_subplot(gs[0, 0])
    divider = make_axes_locatable(ax_map)
    cax = divider.append_axes("right", size="5%", pad=0.05)

    gdf_all.plot(ax=ax_map, column='up', cmap=cmap, norm=norm, markersize=5, legend=True, cax=cax)
    gpd.GeoDataFrame(
        geometry=gpd.points_from_xy([ponto[0]], [ponto[1]]), crs="EPSG:3035"
    ).to_crs(3857).plot(ax=ax_map, color='red', edgecolor='black', markersize=100)
    ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
    ax_map.set_xlim(xmin, xmax)
    ax_map.set_ylim(ymin, ymax)
    ax_map.set_title(f"Ponto ({ponto[0]:.0f}, {ponto[1]:.0f})")
    ax_map.set_axis_off()

    # --- Série temporal ---
    ax_series = fig.add_subplot(gs[0, 1])
    ax_series.plot(datas_clean, up_clean, color="blue", label="up (mm)")
    ax_series.axhline(0, color='gray', linestyle='--', linewidth=0.7)

    ax_nivel = ax_series.twinx()
    ax_nivel.plot(datas_clean, nivel_clean, color="green", label="nível (m)")
    ax_nivel.set_ylabel("Nível (m)", color="green")
    ax_nivel.tick_params(axis='y', labelcolor="green")

    ax_temp = ax_series.twinx()
    ax_temp.spines["right"].set_position(("axes", 1.1))  # desloca eixo à direita
    ax_temp.plot(datas_clean, temp_clean, color="orange", label="temperatura (°C)")
    ax_temp.set_ylabel("Temp (°C)", color="orange")
    ax_temp.tick_params(axis='y', labelcolor="orange")

    ax_series.set_title("Série Temporal: up vs Nível vs Temperatura")
    ax_series.set_ylabel("Deslocamento (mm)")
    ax_series.set_xlabel("Data")
    ax_series.grid(True)

    # --- Dispersão up vs nível ---
    ax_corr_nivel = fig.add_subplot(gs[1, :])
    sns.scatterplot(x=up_clean, y=nivel_clean, color="blue", label="Pontos", ax=ax_corr_nivel)
    sns.regplot(x=up_clean, y=nivel_clean, scatter=False, color="red", line_kws={"lw": 2, "ls": "--"}, ax=ax_corr_nivel)
    ax_corr_nivel.set_title(f'Correlação up vs nível: r = {corr_nivel:.2f}')
    ax_corr_nivel.set_xlabel("Deslocamento (mm)")
    ax_corr_nivel.set_ylabel("Nível (m)")
    ax_corr_nivel.grid(True)

    # --- Dispersão up vs temperatura ---
    ax_corr_temp = fig.add_subplot(gs[2, :])
    sns.scatterplot(x=up_clean, y=temp_clean, color="purple", label="Pontos", ax=ax_corr_temp)
    sns.regplot(x=up_clean, y=temp_clean, scatter=False, color="orange", line_kws={"lw": 2, "ls": "--"}, ax=ax_corr_temp)
    ax_corr_temp.set_title(f'Correlação up vs temperatura: r = {corr_temp:.2f}')
    ax_corr_temp.set_xlabel("Deslocamento (mm)")
    ax_corr_temp.set_ylabel("Temperatura (°C)")
    ax_corr_temp.grid(True)

    plt.tight_layout()
    plt.show()

# ----- Gerar DataFrame com os resultados -----
df_resultados = pd.DataFrame(tabela_resultados)

# Selecionar top_n pontos com maior correlação (em valor absoluto) com nível
top_corr_nivel = df_resultados.reindex(df_resultados['correlacao_nivel'].abs().sort_values(ascending=False).index).head(top_n)

# Selecionar top_n pontos com maior correlação (em valor absoluto) com temperatura
top_corr_temp = df_resultados.reindex(df_resultados['correlacao_temp'].abs().sort_values(ascending=False).index).head(top_n)

# Combinar os dois e eliminar duplicados
pontos_escolhidos = pd.concat([top_corr_nivel, top_corr_temp]).drop_duplicates(subset=['easting', 'northing'])

print("Pontos escolhidos para plot:")
print(pontos_escolhidos[['easting', 'northing', 'correlacao_nivel', 'correlacao_temp']])

# ----------- Plots para os pontos selecionados -------------
for idx, row in pontos_escolhidos.iterrows():
    ponto = (row['easting'], row['northing'])
    serie = df_final[
        (df_final['easting'] == ponto[0]) & (df_final['northing'] == ponto[1])
    ].sort_values('date')

    datas = serie['date']
    up_values = serie['up']

    nivel_alinhado = df_nivel.reindex(datas, method='nearest')['nivel'].values
    temp_alinhado = df_temp.reindex(datas, method='nearest')['med'].values  # ou 'max'/'min'

    valid = ~np.isnan(up_values) & ~np.isnan(nivel_alinhado) & ~np.isnan(temp_alinhado)

    if valid.sum() < 3:
        continue

    up_clean = up_values[valid]
    nivel_clean = nivel_alinhado[valid]
    temp_clean = temp_alinhado[valid]
    datas_clean = datas[valid]

    corr_nivel = np.corrcoef(up_clean, nivel_clean)[0, 1]
    corr_temp = np.corrcoef(up_clean, temp_clean)[0, 1]

    # ---------- Plot ----------
    fig = plt.figure(figsize=(20, 12))
    gs = gridspec.GridSpec(3, 2, height_ratios=[2, 1.2, 1.2], width_ratios=[1.2, 1])

    # --- Série temporal ---
    ax_series = fig.add_subplot(gs[0, :])
    ax_series.plot(datas_clean, up_clean, color="blue", label="up (mm)")
    ax_series.axhline(0, color='gray', linestyle='--', linewidth=0.7)

    ax_nivel = ax_series.twinx()
    ax_nivel.plot(datas_clean, nivel_clean, color="green", label="nível (m)")
    ax_nivel.set_ylabel("Nível (m)", color="green")
    ax_nivel.tick_params(axis='y', labelcolor="green")

    ax_temp = ax_series.twinx()
    ax_temp.spines["right"].set_position(("axes", 1.1))
    ax_temp.plot(datas_clean, temp_clean, color="orange", label="temperatura (°C)")
    ax_temp.set_ylabel("Temp (°C)", color="orange")
    ax_temp.tick_params(axis='y', labelcolor="orange")

    ax_series.set_title(f"Ponto ({ponto[0]:.0f}, {ponto[1]:.0f}) - Série Temporal")
    ax_series.set_ylabel("Deslocamento (mm)")
    ax_series.set_xlabel("Data")
    ax_series.grid(True)

    # --- Dispersão up vs nível ---
    ax_corr_nivel = fig.add_subplot(gs[1, :])
    sns.scatterplot(x=up_clean, y=nivel_clean, color="blue", ax=ax_corr_nivel)
    sns.regplot(x=up_clean, y=nivel_clean, scatter=False, color="red", line_kws={"lw": 2, "ls": "--"}, ax=ax_corr_nivel)
    ax_corr_nivel.set_title(f'Correlação up vs nível: r = {corr_nivel:.2f}')
    ax_corr_nivel.set_xlabel("Deslocamento (mm)")
    ax_corr_nivel.set_ylabel("Nível (m)")
    ax_corr_nivel.grid(True)

    # --- Dispersão up vs temperatura ---
    ax_corr_temp = fig.add_subplot(gs[2, :])
    sns.scatterplot(x=up_clean, y=temp_clean, color="purple", ax=ax_corr_temp)
    sns.regplot(x=up_clean, y=temp_clean, scatter=False, color="orange", line_kws={"lw": 2, "ls": "--"}, ax=ax_corr_temp)
    ax_corr_temp.set_title(f'Correlação up vs temperatura: r = {corr_temp:.2f}')
    ax_corr_temp.set_xlabel("Deslocamento (mm)")
    ax_corr_temp.set_ylabel("Temperatura (°C)")
    ax_corr_temp.grid(True)

    plt.tight_layout()
    plt.show()


In [ ]:
# --- Importações adicionais ---
from sklearn.metrics import r2_score
import statsmodels.api as sm
from scipy.signal import correlate

import matplotlib.colors as colors
from matplotlib.cm import ScalarMappable
from scipy.interpolate import griddata
import contextily as ctx
import seaborn as sns
import matplotlib.gridspec as gridspec
from mpl_toolkits.axes_grid1 import make_axes_locatable

# ----------- Parâmetros editáveis ------------
componente = "up"
vmin, vcenter, vmax = -5, 0, 5  # mm
top_n = 2  # Número de pontos mais instáveis
usar_colormap_reverso = True

# ----------- Leitura dados ---------
#df_asc = pd.read_csv("...")  # caminho ASC
#df_desc = pd.read_csv("...")  # caminho DESC
df_nivel = pd.read_excel("data/nivel.xlsx")
df_nivel['data'] = pd.to_datetime(df_nivel['data'])
df_nivel.set_index('data', inplace=True)

# ----------- Filtro espacial ---------
def filtro_alqueva(df):
    return df[
        (df['northing'] >= 1855050) & (df['northing'] <= 1855850) &
        (df['easting'] >= 2792250) & (df['easting'] <= 2793250)
    ]
df_asc = filtro_alqueva(df_asc)
df_desc = filtro_alqueva(df_desc)

# ----------- Pré-processamento ------------
def melt(df, comp):
    static = ['pid', 'easting', 'northing']
    datas = [c for c in df.columns if c.startswith('20')]
    df_temp = df[static + datas].copy()
    df_melt = df_temp.melt(id_vars=static, var_name='date', value_name=comp)
    df_melt['date'] = pd.to_datetime(df_melt['date'], format='%Y%m%d')
    return df_melt

def get_componentes(df, prefix):
    up = melt(df, f'up_{prefix}')
    east = melt(df, f'east_{prefix}')
    north = melt(df, f'north_{prefix}')
    merged = up.merge(east, on=['pid', 'easting', 'northing', 'date'])
    merged = merged.merge(north, on=['pid', 'easting', 'northing', 'date'])
    return merged

asc = get_componentes(df_asc, 'asc')
desc = get_componentes(df_desc, 'desc')

datas_comuns = np.intersect1d(asc['date'].unique(), desc['date'].unique())

# ----------- Interpolação espacial e média ------------
resultados = []
for data in datas_comuns:
    asc_data = asc[asc['date'] == data]
    desc_data = desc[desc['date'] == data]
    pontos_asc = asc_data[['easting', 'northing']].values
    pontos_desc = desc_data[['easting', 'northing']].values

    interp = {
        comp: griddata(pontos_asc, asc_data[comp].values, pontos_desc, method='linear')
        for comp in ['up_asc', 'east_asc', 'north_asc']
    }

    df_comb = pd.DataFrame({
        'easting': desc_data['easting'].values,
        'northing': desc_data['northing'].values,
        'date': data,
        'up': np.nanmean([interp['up_asc'], desc_data['up_desc'].values], axis=0),
        'east': np.nanmean([interp['east_asc'], desc_data['east_desc'].values], axis=0),
        'north': np.nanmean([interp['north_asc'], desc_data['north_desc'].values], axis=0),
    })

    resultados.append(df_comb)

df_final = pd.concat(resultados, ignore_index=True)

# ----------- Média por ponto ------------
df_mean = df_final.groupby(['easting', 'northing'], as_index=False).mean(numeric_only=True)
df_mean['std_up'] = df_final.groupby(['easting', 'northing'])['up'].std().values  # Desvio padrão

# ----------- Top N pontos mais instáveis ------------
df_top = df_mean.nlargest(top_n, 'std_up')

# ----------- GeoDataFrame ------------
gdf_all = gpd.GeoDataFrame(df_mean, geometry=gpd.points_from_xy(df_mean.easting, df_mean.northing), crs="EPSG:3035").to_crs(epsg=3857)
gdf_top = gpd.GeoDataFrame(df_top, geometry=gpd.points_from_xy(df_top.easting, df_top.northing), crs="EPSG:3035").to_crs(epsg=3857)

# ----------- Normalização para mapa ------------
norm = colors.TwoSlopeNorm(vmin=vmin, vcenter=vcenter, vmax=vmax)
xmin, ymin, xmax, ymax = gdf_all.total_bounds
cmap = plt.cm.jet.reversed() if usar_colormap_reverso else plt.cm.jet

# ... [o teu código anterior permanece igual até ao loop for _, row in df_top.iterrows()] ...

for _, row in df_top.iterrows():
    ponto = (row['easting'], row['northing'])
    serie = df_final[(df_final['easting'] == ponto[0]) & (df_final['northing'] == ponto[1])]
    serie = serie.sort_values('date')
    datas = serie['date']
    up_values = serie['up']

    # Alinhar com nível
    nivel_alinhado = df_nivel.reindex(datas, method='nearest')['nivel'].values
    valid = ~np.isnan(up_values) & ~np.isnan(nivel_alinhado)
    up_clean = up_values[valid]
    nivel_clean = nivel_alinhado[valid]
    datas_clean = datas[valid]

    # Correlação e tendência
    corr_value = np.corrcoef(up_clean, nivel_clean)[0, 1]
    coef = np.polyfit(datas_clean.map(pd.Timestamp.toordinal), up_clean, deg=1)

    # --- R² da regressão linear ---
    r2 = r2_score(nivel_clean, np.polyval(coef, datas_clean.map(pd.Timestamp.toordinal)))

    # --- Regressão com p-value ---
    X = sm.add_constant(datas_clean.map(pd.Timestamp.toordinal))
    model = sm.OLS(up_clean, X).fit()
    p_value = model.pvalues[1]
    slope = model.params[1]

    # --- Correlação cruzada (lag) ---
    # Descomenta se quiseres usar esta análise:
    lags = np.arange(-30, 31)
    corrs = [np.corrcoef(
         up_clean.shift(-lag).dropna(), 
         nivel_clean[:len(up_clean.shift(-lag).dropna())]
     )[0, 1] if len(up_clean.shift(-lag).dropna()) > 10 else np.nan for lag in lags]
    best_lag = lags[np.nanargmax(corrs)]

    # ----------- Plot melhorado com gridspec ------------
    fig = plt.figure(figsize=(18, 10))
    gs = gridspec.GridSpec(2, 2, height_ratios=[2, 1], width_ratios=[1.3, 1])

    # --- Mapa ---
    ax_map = fig.add_subplot(gs[0, 0])
    divider = make_axes_locatable(ax_map)
    cax = divider.append_axes("right", size="5%", pad=0.05)

    gdf_all.plot(ax=ax_map, column='up', cmap=cmap, norm=norm, markersize=5, legend=True, cax=cax)
    gpd.GeoDataFrame(
        geometry=gpd.points_from_xy([row['easting']], [row['northing']]),
        crs="EPSG:3035"
    ).to_crs(3857).plot(ax=ax_map, color='red', edgecolor='black', markersize=100)

    ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
    ax_map.set_xlim(xmin, xmax)
    ax_map.set_ylim(ymin, ymax)
    ax_map.set_title(f"Ponto mais instável | Média: {row['up']:.2f} mm")
    ax_map.set_axis_off()

    # --- Série temporal ---
    ax_series = fig.add_subplot(gs[0, 1])
    ax_series.plot(datas, up_values, color="blue", label="up")
    ax_series.axhline(0, color='gray', linestyle='--', linewidth=0.7)
    ax_series.set_title("Série Temporal: up vs Nível")
    ax_series.set_ylabel("Deslocamento (mm)")
    ax_series.set_xlabel("Data")
    ax_series.grid(True)

    if len(nivel_clean) > 0:
        ax2 = ax_series.twinx()
        ax2.plot(datas, nivel_alinhado, color="green", label="Nível")
        ax2.set_ylabel("Nível (m)", color="green")
        ax2.tick_params(axis='y', labelcolor="green")
        ax_series.legend(loc='upper left')
        ax2.legend(loc='upper right')

    # --- Dispersão + regressão ---
    ax_corr = fig.add_subplot(gs[1, :])
    sns.scatterplot(x=up_clean, y=nivel_clean, color="blue", label="Pontos", ax=ax_corr)
    sns.regplot(x=up_clean, y=nivel_clean, scatter=False, color="red", line_kws={"lw": 2, "ls": "--"}, ax=ax_corr)

    ax_corr.set_title(
        f'Correlação = {corr_value:.2f} | R² = {r2:.2f} | Tendência = {slope:.4f} mm/dia (p = {p_value:.3f})'
        # + f' | Lag Máx Corr: {best_lag} dias'  # <- ativa se usares lag
    )
    ax_corr.set_xlabel("Deslocamento (mm)")
    ax_corr.set_ylabel("Nível (m)")
    ax_corr.grid(True)

    plt.tight_layout()
    plt.show()


In [ ]:
# --- Importações adicionais ---
from sklearn.metrics import r2_score
import statsmodels.api as sm
from scipy.signal import correlate

import matplotlib.colors as colors
from matplotlib.cm import ScalarMappable
from scipy.interpolate import griddata
import contextily as ctx
import seaborn as sns
import matplotlib.gridspec as gridspec
from mpl_toolkits.axes_grid1 import make_axes_locatable

# ----------- Parâmetros editáveis ------------
componente = "up"
vmin, vcenter, vmax = -5, 0, 5  # mm
top_n = 2  # Número de pontos mais instáveis
usar_colormap_reverso = True

# ----------- Leitura dados ---------
#df_asc = pd.read_csv("...")  # caminho ASC
#df_desc = pd.read_csv("...")  # caminho DESC
df_nivel = pd.read_excel("data/nivel.xlsx")
df_nivel['data'] = pd.to_datetime(df_nivel['data'])
df_nivel.set_index('data', inplace=True)

# ----------- Filtro espacial ---------
def filtro_alqueva(df):
    return df[
        (df['northing'] >= 1855050) & (df['northing'] <= 1855850) &
        (df['easting'] >= 2792250) & (df['easting'] <= 2793250)
    ]
df_asc = filtro_alqueva(df_asc)
df_desc = filtro_alqueva(df_desc)

# ----------- Pré-processamento ------------
def melt(df, comp):
    static = ['pid', 'easting', 'northing']
    datas = [c for c in df.columns if c.startswith('20')]
    df_temp = df[static + datas].copy()
    df_melt = df_temp.melt(id_vars=static, var_name='date', value_name=comp)
    df_melt['date'] = pd.to_datetime(df_melt['date'], format='%Y%m%d')
    return df_melt

def get_componentes(df, prefix):
    up = melt(df, f'up_{prefix}')
    east = melt(df, f'east_{prefix}')
    north = melt(df, f'north_{prefix}')
    merged = up.merge(east, on=['pid', 'easting', 'northing', 'date'])
    merged = merged.merge(north, on=['pid', 'easting', 'northing', 'date'])
    return merged

asc = get_componentes(df_asc, 'asc')
desc = get_componentes(df_desc, 'desc')

datas_comuns = np.intersect1d(asc['date'].unique(), desc['date'].unique())

# ----------- Interpolação espacial e média ------------
resultados = []
for data in datas_comuns:
    asc_data = asc[asc['date'] == data]
    desc_data = desc[desc['date'] == data]
    pontos_asc = asc_data[['easting', 'northing']].values
    pontos_desc = desc_data[['easting', 'northing']].values

    interp = {
        comp: griddata(pontos_asc, asc_data[comp].values, pontos_desc, method='linear')
        for comp in ['up_asc', 'east_asc', 'north_asc']
    }

    df_comb = pd.DataFrame({
        'easting': desc_data['easting'].values,
        'northing': desc_data['northing'].values,
        'date': data,
        'up': np.nanmean([interp['up_asc'], desc_data['up_desc'].values], axis=0),
        'east': np.nanmean([interp['east_asc'], desc_data['east_desc'].values], axis=0),
        'north': np.nanmean([interp['north_asc'], desc_data['north_desc'].values], axis=0),
    })

    resultados.append(df_comb)

df_final = pd.concat(resultados, ignore_index=True)

# ----------- Média por ponto ------------
df_mean = df_final.groupby(['easting', 'northing'], as_index=False).mean(numeric_only=True)
df_mean['std_up'] = df_final.groupby(['easting', 'northing'])['up'].std().values  # Desvio padrão

# ----------- Top N pontos mais instáveis ------------
df_top = df_mean.nlargest(top_n, 'std_up')

# ----------- GeoDataFrame ------------
gdf_all = gpd.GeoDataFrame(df_mean, geometry=gpd.points_from_xy(df_mean.easting, df_mean.northing), crs="EPSG:3035").to_crs(epsg=3857)
gdf_top = gpd.GeoDataFrame(df_top, geometry=gpd.points_from_xy(df_top.easting, df_top.northing), crs="EPSG:3035").to_crs(epsg=3857)

# ----------- Normalização para mapa ------------
norm = colors.TwoSlopeNorm(vmin=vmin, vcenter=vcenter, vmax=vmax)
xmin, ymin, xmax, ymax = gdf_all.total_bounds
cmap = plt.cm.jet.reversed() if usar_colormap_reverso else plt.cm.jet


# ... [o teu código anterior permanece igual até ao loop for _, row in df_top.iterrows()] ...
import os

output_dir = "resultados"
os.makedirs(output_dir, exist_ok=True)

for idx, row in df_top.iterrows():
    ponto = (row['easting'], row['northing'])
    serie = df_final[(df_final['easting'] == ponto[0]) & (df_final['northing'] == ponto[1])]
    serie = serie.sort_values('date')
    datas = serie['date']
    up_values = serie['up']

    # Alinhar com nível
    nivel_alinhado = df_nivel.reindex(datas, method='nearest')['nivel'].values
    valid = ~np.isnan(up_values) & ~np.isnan(nivel_alinhado)
    up_clean = up_values[valid]
    nivel_clean = nivel_alinhado[valid]
    datas_clean = datas[valid]

    # Correlação e regressão
    corr_value = np.corrcoef(up_clean, nivel_clean)[0, 1] if len(up_clean) > 2 else np.nan
    coef = np.polyfit(datas_clean.map(pd.Timestamp.toordinal), up_clean, deg=1) if len(up_clean) > 1 else [np.nan, np.nan]
    r2 = r2_score(nivel_clean, np.polyval(coef, datas_clean.map(pd.Timestamp.toordinal))) if len(up_clean) > 1 else np.nan
    X = sm.add_constant(datas_clean.map(pd.Timestamp.toordinal))
    model = sm.OLS(up_clean, X).fit() if len(up_clean) > 2 else None
    p_value = model.pvalues[1] if model is not None else np.nan
    slope = model.params[1] if model is not None else np.nan

    # Nome do ficheiro
    nome_base = f"pt_{int(ponto[0])}_{int(ponto[1])}"
    output_path = os.path.join(output_dir, f"{nome_base}.png")

    # ----------- Plot com gridspec ------------
    fig = plt.figure(figsize=(18, 10))
    gs = gridspec.GridSpec(2, 2, height_ratios=[2, 1], width_ratios=[1.3, 1])

    # --- Mapa ---
    ax_map = fig.add_subplot(gs[0, 0])
    divider = make_axes_locatable(ax_map)
    cax = divider.append_axes("right", size="5%", pad=0.05)

    gdf_all.plot(ax=ax_map, column='up', cmap=cmap, norm=norm, markersize=5, legend=True, cax=cax)
    gpd.GeoDataFrame(
        geometry=gpd.points_from_xy([row['easting']], [row['northing']]),
        crs="EPSG:3035"
    ).to_crs(3857).plot(ax=ax_map, color='red', edgecolor='black', markersize=100)

    # Inserir métricas no mapa
    ax_map.text(
        0.02, 0.95,
        f"Corr: {corr_value:.2f}\nR²: {r2:.2f}\nTendência: {slope:.4f} mm/dia\np: {p_value:.3f}",
        transform=ax_map.transAxes,
        fontsize=10,
        color='white',
        bbox=dict(facecolor='black', alpha=0.6, boxstyle='round,pad=0.5')
    )

    ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
    ax_map.set_xlim(xmin, xmax)
    ax_map.set_ylim(ymin, ymax)
    ax_map.set_title(f"Ponto mais instável | Média: {row['up']:.2f} mm")
    ax_map.set_axis_off()

    # --- Série temporal ---
    ax_series = fig.add_subplot(gs[0, 1])
    ax_series.plot(datas, up_values, color="blue", label="up")
    ax_series.axhline(0, color='gray', linestyle='--', linewidth=0.7)
    ax_series.set_title("Série Temporal: up vs Nível")
    ax_series.set_ylabel("Deslocamento (mm)")
    ax_series.set_xlabel("Data")
    ax_series.grid(True)

    if len(nivel_clean) > 0:
        ax2 = ax_series.twinx()
        ax2.plot(datas, nivel_alinhado, color="green", label="Nível")
        ax2.set_ylabel("Nível (m)", color="green")
        ax2.tick_params(axis='y', labelcolor="green")
        ax_series.legend(loc='upper left')
        ax2.legend(loc='upper right')

    # --- Dispersão + regressão ---
    ax_corr = fig.add_subplot(gs[1, :])
    sns.scatterplot(x=up_clean, y=nivel_clean, color="blue", label="Pontos", ax=ax_corr)
    sns.regplot(x=up_clean, y=nivel_clean, scatter=False, color="red", line_kws={"lw": 2, "ls": "--"}, ax=ax_corr)
    ax_corr.set_title(f'Correlação = {corr_value:.2f} | R² = {r2:.2f} | Tendência = {slope:.4f} mm/dia (p = {p_value:.3f})')
    ax_corr.set_xlabel("Deslocamento (mm)")
    ax_corr.set_ylabel("Nível (m)")
    ax_corr.grid(True)

    plt.tight_layout()
    plt.savefig(output_path, dpi=300)  # Guardar figura
    plt.close()


### K-Means

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx
from sklearn.cluster import KMeans

# === 1. Pivot para cada componente ===
pivot_up = df_final.pivot(index=['easting', 'northing'], columns='date', values='up')
pivot_east = df_final.pivot(index=['easting', 'northing'], columns='date', values='east')
pivot_north = df_final.pivot(index=['easting', 'northing'], columns='date', values='north')

# === 2. Concatenar os três componentes ===
X = pd.concat([pivot_up, pivot_east, pivot_north], axis=1)

# === 3. Tratar NaNs ===
X = X.fillna(method='ffill', axis=1).fillna(method='bfill', axis=1).fillna(0)

# === 4. Clustering com KMeans ===
n_clusters = 4
kmeans = KMeans(n_clusters=n_clusters, random_state=42)
labels = kmeans.fit_predict(X)
X['cluster'] = labels

# === 5. Criar GeoDataFrame para o mapa ===
df_coords = pivot_up.reset_index()[['easting', 'northing']]
df_coords['cluster'] = labels
gdf_clusters = gpd.GeoDataFrame(
    df_coords,
    geometry=gpd.points_from_xy(df_coords['easting'], df_coords['northing']),
    crs="EPSG:3035"
)

# === 6. Plot mapa dos clusters ===
fig, ax = plt.subplots(1, 2, figsize=(16, 6))

# --- Mapa ---
gdf_clusters.to_crs(epsg=3857).plot(
    ax=ax[0],
    column='cluster',
    categorical=True,
    legend=True,
    cmap='tab10',
    markersize=20
)
ctx.add_basemap(ax[0], source=ctx.providers.Esri.WorldImagery)
ax[0].set_title("Clusters de deslocamento (KMeans)")
ax[0].set_axis_off()

# === 7. Gráfico das médias das séries UP por cluster ===
for cluster_id in range(n_clusters):
    idx = X['cluster'] == cluster_id
    media_cluster = pivot_up[idx].mean(axis=0)
    ax[1].plot(media_cluster.index, media_cluster.values, label=f'Cluster {cluster_id}')

ax[1].set_xlabel("Data")
ax[1].set_ylabel("Deslocamento UP (mm)")
ax[1].set_title("Média da série temporal UP por cluster")
ax[1].legend()
plt.tight_layout()
plt.show()


### Interpolador IDW:
- IDW pode ser computacionalmente mais pesado do que griddata, especialmente com muitos pontos.
- É sensível à escolha da potência (power). Normalmente entre 1.5 e 3 funciona bem.
- Não extrapola mal como o linear — mas pode alisar demasiado os dados se os pontos estiverem espaçados.

Abaixo tens uma versão do teu código com IDW no lugar de griddata, mantendo toda a lógica original, mas trocando apenas o método de interpolação espacial (ponto 4). Tudo o resto é exatamente igual.

O ficheiro final será salvo como: alqueva_comb_espacial_idw.csv

O parâmetro power=2 na função idw_interpolation() é ajustável. Quanto maior, mais peso é dado a pontos próximos.

Se quiseres comparar com o método linear, agora tens os dois scripts — basta mudar o nome do ficheiro no final e sobrepor os resultados.

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from matplotlib.cm import ScalarMappable
import contextily as ctx

# ======== PARÂMETROS EDITÁVEIS =========
componente = "up"  # opções: 'up', 'east', 'north'
vmin = -5         # mínimo da escala (mm)
vcenter = 0       # centro da escala (normalmente zero)
vmax = 5          # máximo da escala (mm)
usar_colormap_reverso = True
# =======================================

# ========== 1. LER E FILTRAR DADOS ==========
def ler_e_filtrar(caminho_csv):
    df = pd.read_csv(caminho_csv)

    # Filtro espacial (Alqueva)
    norte_min, norte_max = 1855050, 1855850
    este_min, este_max = 2792250, 2793250

    df = df[
        (df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
        (df['easting'] >= este_min) & (df['easting'] <= este_max)
    ]

    return df

df_asc = ler_e_filtrar("data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv")
df_desc = ler_e_filtrar("data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv")

# ========== 2. DERRETER COMPONENTES ==========
def melt(df, comp):
    static = ['pid', 'easting', 'northing']
    dates = [c for c in df.columns if c.startswith('20')]
    df_temp = df[static + dates].copy()
    df_melt = df_temp.melt(id_vars=static, var_name='date', value_name=comp)
    df_melt['date'] = pd.to_datetime(df_melt['date'], format='%Y%m%d')
    return df_melt

def get_componentes(df, prefix):
    up = melt(df, f'up_{prefix}')
    east = melt(df, f'east_{prefix}')
    north = melt(df, f'north_{prefix}')
    merged = up.merge(east, on=['pid', 'easting', 'northing', 'date'])
    merged = merged.merge(north, on=['pid', 'easting', 'northing', 'date'])
    return merged

asc = get_componentes(df_asc, 'asc')
desc = get_componentes(df_desc, 'desc')

# ========== 3. DATAS COMUNS ==========
datas_comuns = np.intersect1d(asc['date'].unique(), desc['date'].unique())

# ========== 4. INTERPOLAÇÃO ESPACIAL (IDW) ==========
def idw_interpolation(xy_known, values_known, xy_target, power=2):
    interpolated = []
    for x0, y0 in xy_target:
        dists = np.sqrt((xy_known[:, 0] - x0)**2 + (xy_known[:, 1] - y0)**2)

        if np.any(dists == 0):
            interpolated.append(values_known[dists == 0][0])
        else:
            weights = 1 / (dists ** power)
            weighted_sum = np.sum(weights * values_known)
            interpolated.append(weighted_sum / np.sum(weights))

    return np.array(interpolated)

resultados = []

for data in datas_comuns:
    asc_data = asc[asc['date'] == data]
    desc_data = desc[desc['date'] == data]

    pontos_asc = asc_data[['easting', 'northing']].values
    pontos_desc = desc_data[['easting', 'northing']].values

    interp = {}
    for comp in ['up_asc', 'east_asc', 'north_asc']:
        interp[comp] = idw_interpolation(
            pontos_asc,
            asc_data[comp].values,
            pontos_desc,
            power=2  # Ajustável
        )

    df_comb = pd.DataFrame({
        'easting': desc_data['easting'].values,
        'northing': desc_data['northing'].values,
        'date': data,
        'up': np.nanmean([interp['up_asc'], desc_data['up_desc'].values], axis=0),
        'east': np.nanmean([interp['east_asc'], desc_data['east_desc'].values], axis=0),
        'north': np.nanmean([interp['north_asc'], desc_data['north_desc'].values], axis=0),
    })

    resultados.append(df_comb)

# ========== 5. SALVAR COMBINADO ==========
df_final = pd.concat(resultados, ignore_index=True)
df_final.to_csv("alqueva_comb_espacial_idw.csv", index=False)

# ========== 6. CARREGAR E PLOTAR ==========
df = pd.read_csv("alqueva_comb_espacial_idw.csv")
df_media = df.groupby(['easting', 'northing'], as_index=False).mean(numeric_only=True)

gdf = gpd.GeoDataFrame(
    df_media,
    geometry=gpd.points_from_xy(df_media['easting'], df_media['northing']),
    crs="EPSG:3035"
)

gdf_web = gdf.to_crs(epsg=3857)

cmap = plt.colormaps.get_cmap('jet')
if usar_colormap_reverso:
    cmap = cmap.reversed()

norma = colors.TwoSlopeNorm(
    vmin=vmin,
    vcenter=vcenter,
    vmax=vmax
)

fig, ax = plt.subplots(figsize=(10, 6))
gdf_web.plot(
    ax=ax,
    column=componente,
    cmap=cmap,
    markersize=10,
    norm=norma,
    legend=False
)

ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)

ax.set_axis_off()
plt.title(f"Deslocamento médio ({componente}) - ASC + DESC (IDW)", fontsize=14)

sm = ScalarMappable(cmap=cmap, norm=norma)
sm._A = []
cbar = plt.colorbar(sm, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label("Deslocamento (mm)", fontsize=12)

plt.tight_layout()
plt.show()



### Comparação interpolador linear com IDW

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib import colors
from matplotlib.cm import ScalarMappable
import contextily as ctx

# === 1. Carregar os dois resultados ===
df_linear = pd.read_csv("alqueva_comb_espacial_linear.csv")
df_idw = pd.read_csv("alqueva_comb_espacial_idw.csv")

# === 2. Calcular médias por ponto ===
df_linear_media = df_linear.groupby(['easting', 'northing'], as_index=False).mean(numeric_only=True)
df_idw_media = df_idw.groupby(['easting', 'northing'], as_index=False).mean(numeric_only=True)

# === 3. Criar GeoDataFrames ===
gdf_linear = gpd.GeoDataFrame(
    df_linear_media,
    geometry=gpd.points_from_xy(df_linear_media['easting'], df_linear_media['northing']),
    crs="EPSG:3035"
)

gdf_idw = gpd.GeoDataFrame(
    df_idw_media,
    geometry=gpd.points_from_xy(df_idw_media['easting'], df_idw_media['northing']),
    crs="EPSG:3035"
)

# Converter para Web Mercator
gdf_linear = gdf_linear.to_crs(epsg=3857)
gdf_idw = gdf_idw.to_crs(epsg=3857)

# === 4. Parâmetros de plot ===
componente = "up"  # ou 'east' ou 'north'
vmin, vmax = -5, 5
cmap = plt.colormaps.get_cmap("jet").reversed()
norma = colors.TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)

# === 5. Figura lado a lado com legenda horizontal ===
fig, axs = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)

for ax, gdf_data, title in zip(
    axs,
    [gdf_linear, gdf_idw],
    ["Interpolação Linear", "Interpolação IDW"]
):
    gdf_data.plot(
        ax=ax,
        column=componente,
        cmap=cmap,
        markersize=10,
        norm=norma,
        legend=False
    )
    ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)
    ax.set_title(title, fontsize=13)
    ax.set_axis_off()

sm = ScalarMappable(cmap=cmap, norm=norma)
sm._A = []
cbar = plt.colorbar(sm, ax=axs, orientation='horizontal', fraction=0.05, pad=0.05)
cbar.set_label(f"Deslocamento {componente.upper()} (mm)", fontsize=12)

plt.show()



### Comparação com interpolador linear com o ortho

In [ ]:
# === 0. Parâmetros configuráveis ===
vmin, vmax = -5, 5  # Limites da escala de deslocamento (mm)
componente = 'up'  # ou 'east'

# === 1. Função para filtrar ===
def ler_e_filtrar(caminho_csv):
    df = pd.read_csv(caminho_csv)
    norte_min, norte_max = 1855050, 1855850
    este_min, este_max = 2792250, 2793250
    df = df[
        (df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
        (df['easting'] >= este_min) & (df['easting'] <= este_max)
    ]
    return df

# === 2. Carregar dados ORTHO filtrados ===
df_ortho_e = ler_e_filtrar("data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv")
df_ortho_u = ler_e_filtrar("data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv")

# === 3. Função para calcular média e criar GeoDataFrame ORTHO ===
def media_ortho(df_ortho, componente_nome):
    datas = [c for c in df_ortho.columns if c.startswith('20')]
    df_ortho['media'] = df_ortho[datas].mean(axis=1, skipna=True)
    gdf = gpd.GeoDataFrame(
        df_ortho[['easting', 'northing', 'media']],
        geometry=gpd.points_from_xy(df_ortho['easting'], df_ortho['northing']),
        crs="EPSG:3035"
    )
    gdf = gdf.rename(columns={'media': componente_nome})
    return gdf

# === 4. Criar gdf_ortho_web com o componente desejado ===
dict_df_ortho = {'up': df_ortho_u, 'east': df_ortho_e}
gdf_ortho = media_ortho(dict_df_ortho[componente], componente)
gdf_ortho_web = gdf_ortho.to_crs(epsg=3857)

# === 5. Carregar dados da fusão ASC+DESC já processados ===
df_fusao = pd.read_csv("alqueva_comb_espacial_linear.csv")
df_media = df_fusao.groupby(['easting', 'northing'], as_index=False).mean(numeric_only=True)
gdf_fusao = gpd.GeoDataFrame(
    df_media,
    geometry=gpd.points_from_xy(df_media['easting'], df_media['northing']),
    crs="EPSG:3035"
)
gdf_web = gdf_fusao.to_crs(epsg=3857)

# === 6. Plot comparativo ===
cmap = plt.colormaps.get_cmap('jet').reversed()
norma = colors.TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)

fig, axs = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)

for ax, gdf_data, title in zip(
    axs,
    [gdf_web, gdf_ortho_web],
    ["Fusão ASC+DESC", "EGMS ORTHO"]
):
    gdf_data.plot(
        ax=ax,
        column=componente,
        cmap=cmap,
        markersize=10,
        norm=norma,
        legend=False
    )
    ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)
    ax.set_title(title, fontsize=13)
    ax.set_axis_off()

sm = ScalarMappable(cmap=cmap, norm=norma)
sm._A = []
cbar = plt.colorbar(sm, ax=axs, orientation='horizontal', fraction=0.05, pad=0.05)
cbar.set_label(f"Deslocamento {componente.upper()} (mm)", fontsize=12)

plt.show()


### Comparação com interpolador IDW com o ortho

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx
from matplotlib import colors
from matplotlib.cm import ScalarMappable

# === 0. Parâmetros configuráveis ===
vmin, vmax = -5, 5  # Limites da escala de deslocamento (mm)
componente = 'up'  # ou 'east'

# === 1. Função para filtrar ===
def ler_e_filtrar(caminho_csv):
    df = pd.read_csv(caminho_csv)
    norte_min, norte_max = 1855050, 1855850
    este_min, este_max = 2792250, 2793250
    df = df[
        (df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
        (df['easting'] >= este_min) & (df['easting'] <= este_max)
    ]
    return df

# === 2. Carregar dados ORTHO filtrados ===
df_ortho_e = ler_e_filtrar("data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv")
df_ortho_u = ler_e_filtrar("data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv")

# === 3. Função para calcular média e criar GeoDataFrame ORTHO ===
def media_ortho(df_ortho, componente_nome):
    datas = [c for c in df_ortho.columns if c.startswith('20')]
    df_ortho['media'] = df_ortho[datas].mean(axis=1, skipna=True)
    gdf = gpd.GeoDataFrame(
        df_ortho[['easting', 'northing', 'media']],
        geometry=gpd.points_from_xy(df_ortho['easting'], df_ortho['northing']),
        crs="EPSG:3035"
    )
    gdf = gdf.rename(columns={'media': componente_nome})
    return gdf

# === 4. Criar gdf_ortho_web com o componente desejado ===
dict_df_ortho = {'up': df_ortho_u, 'east': df_ortho_e}
gdf_ortho = media_ortho(dict_df_ortho[componente], componente)
gdf_ortho_web = gdf_ortho.to_crs(epsg=3857)

# === 5. Carregar dados IDW já processados ===
df_idw = pd.read_csv("alqueva_comb_espacial_idw.csv")
df_idw_media = df_idw.groupby(['easting', 'northing'], as_index=False).mean(numeric_only=True)
gdf_idw = gpd.GeoDataFrame(
    df_idw_media,
    geometry=gpd.points_from_xy(df_idw_media['easting'], df_idw_media['northing']),
    crs="EPSG:3035"
)
gdf_idw_web = gdf_idw.to_crs(epsg=3857)

# === 6. Plot comparativo ===
cmap = plt.colormaps.get_cmap('jet').reversed()
norma = colors.TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)

fig, axs = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)

for ax, gdf_data, title in zip(
    axs,
    [gdf_idw_web, gdf_ortho_web],
    ["Interpolação IDW", "EGMS ORTHO"]
):
    gdf_data.plot(
        ax=ax,
        column=componente,
        cmap=cmap,
        markersize=10,
        norm=norma,
        legend=False
    )
    ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)
    ax.set_title(title, fontsize=13)
    ax.set_axis_off()

sm = ScalarMappable(cmap=cmap, norm=norma)
sm._A = []
cbar = plt.colorbar(sm, ax=axs, orientation='horizontal', fraction=0.07, pad=0.05)
cbar.set_label(f"Deslocamento {componente.upper()} (mm)", fontsize=12)

plt.suptitle("Comparação entre Interpolação IDW e ORTHO", fontsize=16)
plt.show()


In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
from sklearn.metrics import mean_absolute_error

# === Parâmetro configurável ===
componente = 'up'  # ou 'east'

# === 1. Garante que os dois GeoDataFrames estão no mesmo CRS ===
gdf_web = gdf_web.to_crs(epsg=3857)
gdf_ortho_web = gdf_ortho_web.to_crs(epsg=3857)

# === 2. Renomeia coluna do ortho para evitar confusão ===
gdf_ortho_web = gdf_ortho_web.rename(columns={componente: f"{componente}_ortho"})

# === 3. Spatial Join (pontos mais próximos dentro de 50 metros) ===
gdf_joined = gpd.sjoin_nearest(
    gdf_web,
    gdf_ortho_web,
    how='inner',
    max_distance=50,   # em metros
    distance_col='dist'
)

print(f"✅ Número de pontos após spatial join: {len(gdf_joined)}")

# === 4. Remove linhas com valores nulos nas colunas de comparação ===
gdf_joined = gdf_joined.dropna(subset=[componente, f"{componente}_ortho"])

# === 5. Extrai valores para comparação ===
val_fusao = gdf_joined[componente].values
val_ortho = gdf_joined[f"{componente}_ortho"].values

# === 6. Calcula métricas ===
correl = np.corrcoef(val_fusao, val_ortho)[0, 1]
diff_media = np.mean(val_fusao - val_ortho)
mae = mean_absolute_error(val_ortho, val_fusao)

# === 7. Mostra os resultados ===
print("\n📊 Comparação entre Fusão ASC+DESC e ORTHO:")
print(f"🔹 Componente: {componente.upper()}")
print(f"🔹 Correlação (Pearson R): {correl:.3f}")
print(f"🔹 Diferença média (fusão - ortho): {diff_media:.2f} mm")
print(f"🔹 Erro absoluto médio (MAE): {mae:.2f} mm")
print(f"🔹 Nº de pontos comparados: {len(val_fusao)}")


## 

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# === Scatter Plot ===
plt.figure(figsize=(8, 8))
plt.scatter(val_ortho, val_fusao, s=20, alpha=0.6, edgecolor='k', label='Pontos')

# Linha de referência y = x
lims = [
    np.min([val_ortho.min(), val_fusao.min()]),
    np.max([val_ortho.max(), val_fusao.max()])
]
plt.plot(lims, lims, 'r--', label='y = x')

# Estilo
plt.xlabel(f'Deslocamento ORTHO ({componente.upper()}) [mm]')
plt.ylabel(f'Deslocamento Fusão ASC+DESC ({componente.upper()}) [mm]')
plt.title(f'Comparação de Deslocamentos - {componente.upper()}')
plt.legend()
plt.grid(True)
plt.axis('square')
plt.xlim(lims)
plt.ylim(lims)
plt.tight_layout()
plt.show()

# 1. Explorar Outliers — pontos com maiores discrepâncias

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import contextily as ctx

df_joined = gdf_joined.copy()

# Extrai coordenadas x e y da geometria
df_joined['easting'] = df_joined.geometry.x
df_joined['northing'] = df_joined.geometry.y

# Calcula o erro absoluto
df_joined['error_abs'] = np.abs(df_joined['up'] - df_joined['up_ortho'])

threshold = 3
outliers = df_joined[df_joined['error_abs'] > threshold]

print(f"Número de outliers (erro > {threshold} mm): {len(outliers)}")
print(outliers[['easting', 'northing', 'up', 'up_ortho', 'error_abs']])

# Plotar outliers no mapa
ax = outliers.plot(
    kind='scatter',
    x='easting',
    y='northing',
    c='error_abs',
    cmap='Reds',
    colorbar=True,
    figsize=(8, 6),
    s=50,
    alpha=0.7,
    edgecolor='k'
)
ctx.add_basemap(ax, crs=df_joined.crs)
ax.set_title("Outliers: Maiores discrepâncias no deslocamento (mm)")
plt.show()



## 2. Testar vários max_distance e ver métricas

In [ ]:
from sklearn.metrics import mean_absolute_error
import pandas as pd

distances = [10, 25, 50, 100, 200]  # em metros
results = []

for dist in distances:
    joined = gpd.sjoin_nearest(
        gdf_web,
        gdf_ortho_web,
        how='inner',
        max_distance=dist,
        distance_col='dist'
    )
    if joined.empty:
        continue
    
    joined['error_abs'] = np.abs(joined['up'] - joined['up_ortho'])
    
    mae = mean_absolute_error(joined['up_ortho'], joined['up'])
    correl = np.corrcoef(joined['up'], joined['up_ortho'])[0, 1]
    
    results.append({
        'max_distance': dist,
        'n_points': len(joined),
        'mae': mae,
        'correlation': correl
    })

df_results = pd.DataFrame(results)
print(df_results)


### 3. Cálculo de RMSE e R²

In [ ]:
import numpy as np
from sklearn.metrics import r2_score

up_fused = df_joined['up'].values
up_ortho = df_joined['up_ortho'].values

rmse = np.sqrt(np.mean((up_ortho - up_fused)**2))
r2 = r2_score(up_ortho, up_fused)

print(f"RMSE: {rmse:.2f} mm")
print(f"R²: {r2:.3f}")

### 1. Mapa de diferenças (heatmap)
- os métodos concordam (diferença próxima de zero) ou divergem (diferenças positivas ou negativas grandes)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as colors

# Calcular a diferença
gdf_joined['dif_up'] = gdf_joined['up'] - gdf_joined['up_ortho']

# Definir norma para o colormap centrado no zero
norma_dif = colors.TwoSlopeNorm(vmin=-5, vcenter=0, vmax=5)

fig, ax = plt.subplots(figsize=(10, 7))
gdf_joined.plot(
    column='dif_up',
    cmap='RdBu_r',
    markersize=30,
    alpha=0.8,
    norm=norma_dif,
    legend=True,
    ax=ax
)

import contextily as ctx
ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)
ax.set_axis_off()
plt.title("Diferença no deslocamento UP (Fusão ASC+DESC - ORTHO)", fontsize=14)
plt.show()


### Histograma da diferença

In [ ]:
import matplotlib.pyplot as plt

gdf_joined['dif_up'] = gdf_joined['up'] - gdf_joined['up_ortho']

plt.figure(figsize=(8, 5))
plt.hist(gdf_joined['dif_up'], bins=50, color='steelblue', edgecolor='black')
plt.axvline(0, color='red', linestyle='--')
plt.title('Histograma das Diferenças (Fusão - ORTHO)')
plt.xlabel('Diferença de deslocamento (mm)')
plt.ylabel('Frequência')
plt.show()


# matriz diss

## Clustering

### Com base no deslocamento médio

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx
from sklearn.cluster import KMeans
from matplotlib import cm
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec

# === Pivot UP ===
pivot_up = df_final.pivot(index=['easting', 'northing'], columns='date', values='up')
tempo = pivot_up.columns
pivot_up = pivot_up.fillna(method='ffill', axis=1).fillna(method='bfill', axis=1).fillna(0)

# === Clustering ===
n_clusters = 4
X = pivot_up.values
kmeans = KMeans(n_clusters=n_clusters, random_state=0)
labels = kmeans.fit_predict(X)

# === GeoDataFrame ===
df_coords = pivot_up.reset_index()[['easting', 'northing']]
df_coords['cluster'] = labels
gdf = gpd.GeoDataFrame(df_coords, geometry=gpd.points_from_xy(df_coords['easting'], df_coords['northing']), crs="EPSG:3035")
gdf_web = gdf.to_crs(epsg=3857)

# === Cores melhores (Set2 ou tab20) ===
cmap = cm.get_cmap('Set2', n_clusters)  # ou 'tab20'
cores_clusters = [cmap(i) for i in range(n_clusters)]

# === Escala comum y ===
y_min = np.nanmin(X)
y_max = np.nanmax(X)

# === Layout ===
ncols = 2
nrows = int(np.ceil(n_clusters / ncols))
fig = plt.figure(figsize=(24, 12 + 3 * nrows))
gs = gridspec.GridSpec(nrows + 1, ncols, height_ratios=[2] + [1]*nrows, hspace=0.4, wspace=0.3)

# --- Mapa ---
ax_mapa = fig.add_subplot(gs[0, :])
gdf_web.plot(ax=ax_mapa, column="cluster", categorical=True, legend=True, cmap="Set2", markersize=25, edgecolor='black', linewidth=0.3)
ctx.add_basemap(ax_mapa, source=ctx.providers.Esri.WorldImagery)
ax_mapa.set_axis_off()
ax_mapa.set_title(f"Clusters espaciais (UP - KMeans)", fontsize=20)

# --- Subplots ---
for cluster_id in range(n_clusters):
    row = (cluster_id // ncols) + 1
    col = cluster_id % ncols
    ax = fig.add_subplot(gs[row, col])

    idx = labels == cluster_id
    series_cluster = X[idx, :]

    for serie in series_cluster:
        ax.plot(tempo, serie, color='lightgray', linewidth=0.7, alpha=0.5)

    media = np.nanmean(series_cluster, axis=0)
    ax.plot(tempo, media, color=cores_clusters[cluster_id], linewidth=3.5, label=f"Cluster {cluster_id}", zorder=3)

    ax.set_title(f"Cluster {cluster_id} ({series_cluster.shape[0]} pontos)", fontsize=14)
    ax.grid(True)
    ax.legend(fontsize=11)
    ax.set_ylabel("Deslocamento UP (mm)", fontsize=12)
    ax.set_ylim(y_min, y_max)
    ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=4, maxticks=8))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))

# Eixo X só na última linha
for ax in fig.get_axes()[-ncols:]:
    ax.set_xlabel("Tempo", fontsize=12)

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx
from sklearn.cluster import KMeans
from matplotlib import cm
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec

# === Pivot UP ===
pivot_up = df_final.pivot(index=['easting', 'northing'], columns='date', values='up')
tempo = pivot_up.columns
pivot_up = pivot_up.fillna(method='ffill', axis=1).fillna(method='bfill', axis=1).fillna(0)

# === Clustering ===
n_clusters = 4
X = pivot_up.values
kmeans = KMeans(n_clusters=n_clusters, random_state=0)
labels = kmeans.fit_predict(X)

# === GeoDataFrame ===
df_coords = pivot_up.reset_index()[['easting', 'northing']]
df_coords['cluster'] = labels
gdf = gpd.GeoDataFrame(df_coords, geometry=gpd.points_from_xy(df_coords['easting'], df_coords['northing']), crs="EPSG:3035")
gdf_web = gdf.to_crs(epsg=3857)

# === Cores melhores (Set2 ou tab20) ===
cmap = cm.get_cmap('Set2', n_clusters)  # ou 'tab20'
cores_clusters = [cmap(i) for i in range(n_clusters)]

# === Escala comum y ===
y_min = np.nanmin(X)
y_max = np.nanmax(X)

# === Layout: 1 linha para mapa + 1 linha com n_clusters colunas para gráficos ===
ncols = n_clusters
nrows = 2

fig = plt.figure(figsize=(5 * ncols, 10))  # ajustar largura proporcional a n_clusters
gs = gridspec.GridSpec(nrows, ncols, height_ratios=[2, 1], hspace=0.4, wspace=0.3)

# --- Mapa ocupa toda a primeira linha (span colunas) ---
ax_mapa = fig.add_subplot(gs[0, :])
gdf_web.plot(ax=ax_mapa, column="cluster", categorical=True, legend=True, cmap="Set2", markersize=25, edgecolor='black', linewidth=0.3)
ctx.add_basemap(ax_mapa, source=ctx.providers.Esri.WorldImagery)
ax_mapa.set_axis_off()
ax_mapa.set_title(f"Clusters espaciais (UP - KMeans)", fontsize=20)

# --- Gráficos na segunda linha, cada cluster em uma coluna ---
for cluster_id in range(n_clusters):
    ax = fig.add_subplot(gs[1, cluster_id])

    idx = labels == cluster_id
    series_cluster = X[idx, :]

    for serie in series_cluster:
        ax.plot(tempo, serie, color='lightgray', linewidth=0.7, alpha=0.5)

    media = np.nanmean(series_cluster, axis=0)
    ax.plot(tempo, media, color=cores_clusters[cluster_id], linewidth=3.5, label=f"Cluster {cluster_id}", zorder=3)

    ax.set_title(f"Cluster {cluster_id} ({series_cluster.shape[0]} pontos)", fontsize=14)
    ax.grid(True)
    ax.legend(fontsize=11)
    ax.set_ylabel("Deslocamento UP (mm)", fontsize=12)
    ax.set_ylim(y_min, y_max)
    ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=4, maxticks=8))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))

    # Mostrar eixo X só no último gráfico para deixar mais limpo
    if cluster_id < n_clusters - 1:
        ax.set_xlabel('')
        ax.tick_params(axis='x', labelbottom=False)

# Eixo X só na última coluna
ax.set_xlabel("Tempo", fontsize=12)

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx
from sklearn.cluster import KMeans
from matplotlib import cm
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec

# === Pivot UP ===
pivot_up = df_final.pivot(index=['easting', 'northing'], columns='date', values='up')
tempo = pivot_up.columns
pivot_up = pivot_up.fillna(method='ffill', axis=1).fillna(method='bfill', axis=1).fillna(0)

# === Clustering ===
n_clusters = 4
X = pivot_up.values
kmeans = KMeans(n_clusters=n_clusters, random_state=0)
labels = kmeans.fit_predict(X)

# === GeoDataFrame ===
df_coords = pivot_up.reset_index()[['easting', 'northing']]
df_coords['cluster'] = labels
gdf = gpd.GeoDataFrame(df_coords, geometry=gpd.points_from_xy(df_coords['easting'], df_coords['northing']), crs="EPSG:3035")
gdf_web = gdf.to_crs(epsg=3857)

# === Cores melhores (Set2 ou tab20) ===
cmap = cm.get_cmap('Set2', n_clusters)  # ou 'tab20'
cores_clusters = [cmap(i) for i in range(n_clusters)]

# === Escala comum y ===
y_min = np.nanmin(X)
y_max = np.nanmax(X)

# === Layout: 1 coluna, n_clusters+1 linhas ===
ncols = 1
nrows = n_clusters + 1

fig = plt.figure(figsize=(12, 4 * nrows))  # Altura proporcional ao número de gráficos
gs = gridspec.GridSpec(nrows, ncols, height_ratios=[2] + [1]*n_clusters, hspace=0.5)

# --- Mapa ocupa primeira linha ---
ax_mapa = fig.add_subplot(gs[0, 0])
gdf_web.plot(ax=ax_mapa, column="cluster", categorical=True, legend=True, cmap="Set2", markersize=25, edgecolor='black', linewidth=0.3)
ctx.add_basemap(ax_mapa, source=ctx.providers.Esri.WorldImagery)
ax_mapa.set_axis_off()
ax_mapa.set_title(f"Clusters espaciais (UP - KMeans)", fontsize=20)

# --- Gráficos em linhas seguintes, mesma coluna ---
for cluster_id in range(n_clusters):
    ax = fig.add_subplot(gs[cluster_id + 1, 0])

    idx = labels == cluster_id
    series_cluster = X[idx, :]

    for serie in series_cluster:
        ax.plot(tempo, serie, color='lightgray', linewidth=0.7, alpha=0.5)

    media = np.nanmean(series_cluster, axis=0)
    ax.plot(tempo, media, color=cores_clusters[cluster_id], linewidth=3.5, label=f"Cluster {cluster_id}", zorder=3)

    ax.set_title(f"Cluster {cluster_id} ({series_cluster.shape[0]} pontos)", fontsize=14)
    ax.grid(True)
    ax.legend(fontsize=11)
    ax.set_ylabel("Deslocamento UP (mm)", fontsize=12)
    ax.set_ylim(y_min, y_max)
    ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=4, maxticks=8))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))

    # Eixo X só no último gráfico
    if cluster_id < n_clusters - 1:
        ax.set_xlabel('')
        ax.tick_params(axis='x', labelbottom=False)

# Eixo X no último gráfico
ax.set_xlabel("Tempo", fontsize=12)

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from matplotlib.cm import ScalarMappable
from sklearn.cluster import KMeans
import contextily as ctx
import numpy as np

# === CONFIGURAÇÕES EDITÁVEIS ===
fonte_dados = df_idw     # ou df_linear
nome_fonte = "IDW"       # ou "Linear"
componente = "up"        # "up", "east", "north"
n_clusters = 4
vmin, vmax = -20, 20     # ESCALA FIXA DE CORES (edite aqui)
decimais_legenda = 1     # NÚMERO DE CASAS DECIMAIS nos limites da legenda

# === MÉDIA TEMPORAL DO COMPONENTE ===
df = fonte_dados.copy()
df[f"mean_{componente}"] = df.filter(like=componente).mean(axis=1)

# === CRIAR GeoDataFrame ===
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df.easting, df.northing),
    crs="EPSG:3035"
)
gdf_web = gdf.to_crs(epsg=3857)

# === KMEANS COM MÉDIA ===
coluna_valor = f"mean_{componente}"
gdf_kmeans = gdf_web.dropna(subset=[coluna_valor]).copy()
X = gdf_kmeans[[coluna_valor]].values

kmeans = KMeans(n_clusters=n_clusters, random_state=42)
gdf_kmeans["cluster"] = kmeans.fit_predict(X)

# === VISUALIZAÇÃO ===
fig, ax = plt.subplots(figsize=(13, 11))

# Usar paleta qualitativa para clusters
cmap = plt.get_cmap('tab10')  # até 10 cores distintas

# Gerar cores para os clusters (mapa cluster_id -> cor)
colors_clusters = [cmap(i) for i in range(n_clusters)]

# Plotar os pontos coloridos pelo cluster
for cluster_id in range(n_clusters):
    subset = gdf_kmeans[gdf_kmeans["cluster"] == cluster_id]
    subset.plot(
        ax=ax,
        color=colors_clusters[cluster_id],
        markersize=12,
        label=f"Cluster {cluster_id}"
    )

ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)
ax.set_axis_off()
ax.set_title(f"KMeans sobre deslocamento médio '{componente.upper()}' [{nome_fonte}]",
             fontsize=15)

# === LEGENDA PERSONALIZADA POR CLUSTER ===
handles = []
for cluster_id in range(n_clusters):
    cluster_data = gdf_kmeans[gdf_kmeans["cluster"] == cluster_id]
    cluster_min = round(cluster_data[coluna_valor].min(), decimais_legenda)
    cluster_max = round(cluster_data[coluna_valor].max(), decimais_legenda)
    cor = colors_clusters[cluster_id]
    handles.append(plt.Line2D(
        [0], [0], marker='o', color='w',
        label=f"Cluster {cluster_id}: {cluster_min} a {cluster_max} m",
        markerfacecolor=cor, markersize=10
    ))

ax.legend(handles=handles, loc='upper left', title="Intervalos por cluster", fontsize=10)

plt.tight_layout()
plt.show()



In [ ]:
df_nivel = pd.read_excel("data/nivel.xlsx")
df_nivel